# Imitator on Kaggle: Self-Contained Pipeline With Parallel Preprocessing

This notebook needs only the raw chess dataset. It writes the required project files into `/kaggle/working/Imitator`, discovers the Kaggle raw PGN layout, preprocesses PGNs in parallel on CPU, and then trains the pretrain and fine-tune models.

Expected dataset layout:
- `/kaggle/input/datasets/yukuanzou/raw-chess-data/data/twic/*.pgn`
- `/kaggle/input/datasets/yukuanzou/raw-chess-data/data/target/bobby_fischer/*.pgn`

Preprocessing is CPU-bound. GPU is used only for model training.


In [ ]:
RUN_PRETRAIN = True
RUN_FINETUNE = True
EXPORT_ARTIFACTS = True
EXPORT_PROCESSED = False

INSTALL_COMPAT_TORCH = True
USE_STREAM_PRETRAIN_STAGE3 = True
STRICT_TARGET_ISOLATION = False
PREPROCESS_WORKERS = 2

DATASET_TAG = "Bobby_Fischer"
TARGET_USERNAME = "Fischer Robert J (USA)"
TARGET_NAME_ALIASES = [
    "Fischer Robert J (USA)",
    "Fischer",
    "Robert James Fischer",
    "Fischer, Robert J",
    "Robert J FISCHER",
]

MODEL_PROFILE = "kaggle_large"

PROFILE_CONFIGS = {
    "baseline": {
        "history_plies": 8,
        "feature_embed_dim": 64,
        "history_hidden_dim": 96,
        "shared_hidden_dim": 128,
        "dropout": 0.10,
        "pretrain_epochs": 8,
        "pretrain_batch_size": 256,
        "pretrain_eval_batch_size": 256,
        "pretrain_grad_accum_steps": 1,
        "pretrain_shuffle_buffer_size": 20000,
        "pretrain_train_eval_max_samples": 50000,
        "pretrain_valid_eval_max_samples": 50000,
        "finetune_epochs": 12,
        "finetune_batch_size": 192,
        "finetune_eval_batch_size": 192,
        "finetune_grad_accum_steps": 1,
        "encoder_learning_rate": 3e-4,
        "head_learning_rate": 1e-3,
        "weight_decay": 1e-5,
    },
    "kaggle_large": {
        "history_plies": 12,
        "feature_embed_dim": 96,
        "history_hidden_dim": 160,
        "shared_hidden_dim": 256,
        "dropout": 0.12,
        "pretrain_epochs": 10,
        "pretrain_batch_size": 384,
        "pretrain_eval_batch_size": 384,
        "pretrain_grad_accum_steps": 1,
        "pretrain_shuffle_buffer_size": 60000,
        "pretrain_train_eval_max_samples": 100000,
        "pretrain_valid_eval_max_samples": 100000,
        "finetune_epochs": 16,
        "finetune_batch_size": 256,
        "finetune_eval_batch_size": 256,
        "finetune_grad_accum_steps": 1,
        "encoder_learning_rate": 2.5e-4,
        "head_learning_rate": 8e-4,
        "weight_decay": 1e-5,
    },
}

PRETRAIN_LEARNING_RATE = 1e-3
PRETRAIN_WEIGHT_DECAY = 1e-5
PRETRAIN_PRINT_EVERY = 25
PRETRAIN_SCAN_PROGRESS_EVERY = 100000
PRETRAIN_STREAM_PROGRESS_EVERY = 200000
FINETUNE_PRINT_EVERY = 20
FINETUNE_USE_PRETRAINED_ENCODERS = True

EXPORT_NAME = "imitator_kaggle_artifacts"
KAGGLE_INPUT_DATA_DIR = "/kaggle/input/datasets/yukuanzou/raw-chess-data"


In [ ]:
if INSTALL_COMPAT_TORCH:
    %pip uninstall -y -q torch torchvision torchaudio
    %pip install -q --no-cache-dir torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 python-chess
else:
    %pip install -q -U python-chess

import os
import platform
import shutil
import torch


def bytes_to_gb(n: int) -> float:
    return n / (1024 ** 3)


disk = shutil.disk_usage("/kaggle/working")
print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print(f"working disk free: {bytes_to_gb(disk.free):.1f} GB")
print("cwd:", os.getcwd())


In [ ]:
import base64
import json
import re
import shutil
from pathlib import Path

EMBEDDED_FILES = {
  "scripts/chess_feature_utils.py": "...",
  "scripts/history_policy_lib.py": "...",
  "scripts/pipeline_config.py": "77u/IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiQ2VudHJhbCBjb25maWd1cmF0aW9uIGFuZCBwYXRoIGhlbHBlcnMgZm9yIHRoZSBpbWl0YXRpb24gcGlwZWxpbmUuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCg0KU0NSSVBUX0RJUiA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQNClBST0pFQ1RfUk9PVCA9IFNDUklQVF9ESVIucGFyZW50DQoNCiMgQWN0aXZlIHRhcmdldC1wbGF5ZXIgZGF0YXNldC4NCkRBVEFTRVRfVEFHID0gIkJvYmJ5X0Zpc2NoZXIiDQpUQVJHRVRfVVNFUk5BTUUgPSAiRmlzY2hlciBSb2JlcnQgSiAoVVNBKSINCiMgT3B0aW9uYWwgYWxpYXNlcyBmb3Igcm9idXN0IG5hbWUgbWF0Y2hpbmcgaW4gUEdOIGhlYWRlcnMuDQpUQVJHRVRfTkFNRV9BTElBU0VTOiBsaXN0W3N0cl0gPSBbJ0Zpc2NoZXIgUm9iZXJ0IEogKFVTQSknLCAnRmlzY2hlcicsICJSb2JlcnQgSmFtZXMgRmlzY2hlciIsICJGaXNjaGVyLCBSb2JlcnQgSiIsICJSb2JlcnQgSiBGSVNDSEVSIl0NCg0KIyBTaGFyZWQgcG9vbCBmb3IgcmVwcmVzZW50YXRpb24gcHJldHJhaW5pbmcuDQpQUkVUUkFJTl9EQVRBU0VUX1RBRyA9ICJwcmV0cmFpbl9tdWx0aSINClBSRVRSQUlOX01FUkdFRF9GSUxFTkFNRSA9ICJwcmV0cmFpbl9tdWx0aV9tZXJnZWQucGduIg0KDQojIERlZGljYXRlZCByb290IGZvciB0YXJnZXQtcGxheWVyIGZpbmUtdHVuaW5nIGRhdGFzZXRzLg0KRklORVRVTkVfUk9PVF9OQU1FID0gImZpbmV0dW5lX3BsYXllcnMiDQoNCiMgT3B0aW9uYWwgc3RhZ2UgMCBtZXJnZSBzZXR0aW5ncy4NCkVOQUJMRV9TVEFHRTBfTUVSR0UgPSBUcnVlDQpQTEFZRVJfUEdOX0dMT0IgPSAiKi5wZ24iDQpQTEFZRVJfUEdOX1JFQ1VSU0lWRSA9IFRydWUNCg0KIyBIaXN0b3J5IHdpbmRvdyBmb3Igc2VxdWVuY2UgZmVhdHVyZXMuDQpISVNUT1JZX1BMSUVTID0gOA0KDQojIFN0YWdlIDMgcHJldHJhaW5pbmcgbW9kZSBpbiBydW5fcHJldHJhaW5fcGlwZWxpbmUucHkuDQojIEZhbHNlOiBpbi1tZW1vcnkgcHJldHJhaW4gc2NyaXB0IChmYXN0ZXIsIGhpZ2hlciBSQU0pDQojIFRydWU6IHN0cmVhbWVkIHByZXRyYWluIHNjcmlwdCAobG93ZXIgUkFNLCBmdWxsLWRhdGEgZnJpZW5kbHkpDQpVU0VfU1RSRUFNX1BSRVRSQUlOX1NUQUdFMyA9IFRydWUNCg0KDQpkZWYgcmF3X2RpcigpIC0+IFBhdGg6DQogICAgcmV0dXJuIFBST0pFQ1RfUk9PVCAvICJkYXRhIiAvICJyYXciDQoNCg0KZGVmIHByZXRyYWluX3Jhd19kaXIoKSAtPiBQYXRoOg0KICAgIHJldHVybiByYXdfZGlyKCkgLyBQUkVUUkFJTl9EQVRBU0VUX1RBRw0KDQoNCmRlZiBwcmV0cmFpbl9wbGF5ZXJfZGlyKHBsYXllcl9zbHVnOiBzdHIpIC0+IFBhdGg6DQogICAgcmV0dXJuIHByZXRyYWluX3Jhd19kaXIoKSAvIHBsYXllcl9zbHVnDQoNCg0KZGVmIHByZXRyYWluX3BsYXllcl9wZ24ocGxheWVyX3NsdWc6IHN0cikgLT4gUGF0aDoNCiAgICByZXR1cm4gcHJldHJhaW5fcmF3X2RpcigpIC8gZiJ7cGxheWVyX3NsdWd9LnBnbiINCg0KDQpkZWYgcHJldHJhaW5fbWVyZ2VkX3BnbigpIC0+IFBhdGg6DQogICAgcmV0dXJuIHByZXRyYWluX3Jhd19kaXIoKSAvIFBSRVRSQUlOX01FUkdFRF9GSUxFTkFNRQ0KDQoNCmRlZiBmaW5ldHVuZV9yYXdfcm9vdCgpIC0+IFBhdGg6DQogICAgcmV0dXJuIHJhd19kaXIoKSAvIEZJTkVUVU5FX1JPT1RfTkFNRQ0KDQoNCmRlZiByYXdfcGduKHRhZzogc3RyID0gREFUQVNFVF9UQUcpIC0+IFBhdGg6DQogICAgcmV0dXJuIGZpbmV0dW5lX3Jhd19yb290KCkgLyBmInt0YWd9LnBnbiINCg0KDQpkZWYgcGxheWVyX3Bnbl9kaXIodGFnOiBzdHIgPSBEQVRBU0VUX1RBRykgLT4gUGF0aDoNCiAgICByZXR1cm4gZmluZXR1bmVfcmF3X3Jvb3QoKSAvIHRhZw0KDQoNCmRlZiBwcm9jZXNzZWRfZGlyKHRhZzogc3RyID0gREFUQVNFVF9UQUcpIC0+IFBhdGg6DQogICAgcmV0dXJuIFBST0pFQ1RfUk9PVCAvICJkYXRhIiAvICJwcm9jZXNzZWQiIC8gdGFnDQoNCg0KZGVmIG1vZGVsc19kaXIodGFnOiBzdHIgPSBEQVRBU0VUX1RBRykgLT4gUGF0aDoNCiAgICByZXR1cm4gUFJPSkVDVF9ST09UIC8gIm1vZGVscyIgLyB0YWcNCg0KDQpkZWYgb3V0cHV0c19kaXIodGFnOiBzdHIgPSBEQVRBU0VUX1RBRykgLT4gUGF0aDoNCiAgICByZXR1cm4gUFJPSkVDVF9ST09UIC8gIm91dHB1dHMiIC8gdGFnDQo=",
  "scripts/script1_parse_multi_player_positions.py": "77u/IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiUGFyc2UgbWVyZ2VkIHByZXRyYWluaW5nIFBHTiBhbmQgZW1pdCBhbGwtcGxheWVyIG1vdmUgc2FtcGxlcyB3aXRoIGhpc3RvcnkuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGpzb24NCmltcG9ydCByZQ0KZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVxdWUNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgRGVxdWUsIERpY3QsIEl0ZXJhdG9yLCBPcHRpb25hbA0KDQppbXBvcnQgY2hlc3MNCmltcG9ydCBjaGVzcy5wZ24NCg0KZnJvbSBjaGVzc19mZWF0dXJlX3V0aWxzIGltcG9ydCAoDQogICAgYXBwbHlfcGllY2VfaWRlbnRpdHlfbW92ZSwNCiAgICBidWlsZF9oaXN0b3J5X2VudHJ5LA0KICAgIGNhbm9uaWNhbF9waWVjZV9zbG90LA0KICAgIGN1cnJlbnRfb3JpZ2luYWxfcGllY2Vfc2xvdF9zcXVhcmVfbWFwLA0KICAgIGN1cnJlbnRfcGllY2VfaWRlbnRpdHksDQogICAgaW5pdGlhbGl6ZV9waWVjZV9pZGVudGl0eV90cmFja2VyLA0KICAgIG5vcm1hbGl6ZV9zcXVhcmUsDQogICAgcGhhc2VfY29kZV9mcm9tX2Z1bGxtb3ZlLA0KKQ0KZnJvbSBwaXBlbGluZV9jb25maWcgaW1wb3J0IEhJU1RPUllfUExJRVMsIFBSRVRSQUlOX0RBVEFTRVRfVEFHLCBwcmV0cmFpbl9tZXJnZWRfcGduLCBwcm9jZXNzZWRfZGlyDQoNCg0KQ09ORklHID0gew0KICAgICJkYXRhc2V0X3RhZyI6IFBSRVRSQUlOX0RBVEFTRVRfVEFHLA0KICAgICJpbnB1dF9wZ24iOiBwcmV0cmFpbl9tZXJnZWRfcGduKCksDQogICAgImhpc3RvcnlfcGxpZXMiOiBISVNUT1JZX1BMSUVTLA0KICAgICJvdXRwdXRfanNvbmwiOiBwcm9jZXNzZWRfZGlyKFBSRVRSQUlOX0RBVEFTRVRfVEFHKSAvICJwb3NpdGlvbnNfaGlzdG9yeS5qc29ubCIsDQogICAgInByb2dyZXNzX2V2ZXJ5X2dhbWVzIjogMTAwLA0KICAgICJ2ZXJib3NlIjogVHJ1ZSwNCn0NCg0KQ0xLX1BBVFRFUk4gPSByZS5jb21waWxlKHIiXFslY2xrXHMrKFswLTldKyk6KFswLTldezJ9KTooWzAtOV17Mn0pXF0iKQ0KDQoNCmRlZiBwYXJzZV9jbG9ja190b19zZWNvbmRzKGNvbW1lbnQ6IHN0cikgLT4gT3B0aW9uYWxbaW50XToNCiAgICAiIiJSZXR1cm4gY2xvY2sgc2Vjb25kcyBmcm9tIFslY2xrIGg6bW06c3NdIHdoZW4gcHJlc2VudC4iIiINCiAgICBpZiBub3QgY29tbWVudDoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBtID0gQ0xLX1BBVFRFUk4uc2VhcmNoKGNvbW1lbnQpDQogICAgaWYgbm90IG06DQogICAgICAgIHJldHVybiBOb25lDQogICAgcmV0dXJuIGludChtLmdyb3VwKDEpKSAqIDM2MDAgKyBpbnQobS5ncm91cCgyKSkgKiA2MCArIGludChtLmdyb3VwKDMpKQ0KDQoNCmRlZiBzYWZlX2ludCh4OiBPcHRpb25hbFtzdHJdKSAtPiBPcHRpb25hbFtpbnRdOg0KICAgICIiIkNvbnZlcnQgb3B0aW9uYWwgc3RyaW5nIHRvIGludC4iIiINCiAgICBpZiB4IGlzIE5vbmU6DQogICAgICAgIHJldHVybiBOb25lDQogICAgeCA9IHguc3RyaXAoKQ0KICAgIGlmIG5vdCB4IG9yIHggPT0gIj8iOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHRyeToNCiAgICAgICAgcmV0dXJuIGludCh4KQ0KICAgIGV4Y2VwdCBWYWx1ZUVycm9yOg0KICAgICAgICByZXR1cm4gTm9uZQ0KDQoNCmRlZiBwcm9tb3Rpb25faW5kZXgobW92ZTogY2hlc3MuTW92ZSkgLT4gaW50Og0KICAgICIiIk1hcCBwcm9tb3Rpb24gcGllY2UgdHlwZSB0byBjb21wYWN0IHByb21vdGlvbiBpZC4iIiINCiAgICBpZiBtb3ZlLnByb21vdGlvbiBpcyBOb25lOg0KICAgICAgICByZXR1cm4gMA0KICAgIHJldHVybiB7Y2hlc3MuS05JR0hUOiAxLCBjaGVzcy5CSVNIT1A6IDIsIGNoZXNzLlJPT0s6IDMsIGNoZXNzLlFVRUVOOiA0fS5nZXQobW92ZS5wcm9tb3Rpb24sIDApDQoNCg0KZGVmIHBsYXllcl9pZChuYW1lOiBzdHIpIC0+IHN0cjoNCiAgICAiIiJDb252ZXJ0IHBsYXllciBkaXNwbGF5IG5hbWUgaW50byBzdGFibGUgbG93ZXJjYXNlIGlkLiIiIg0KICAgIGNsZWFuZWQgPSAiICIuam9pbihuYW1lLnN0cmlwKCkuc3BsaXQoKSkNCiAgICByZXR1cm4gY2xlYW5lZC5sb3dlcigpIGlmIGNsZWFuZWQgZWxzZSAidW5rbm93biINCg0KDQpkZWYgaXRlcl9hbGxfcGxheWVyX3Bvc2l0aW9ucyhwZ25fcGF0aDogUGF0aCwgaGlzdG9yeV9wbGllczogaW50KSAtPiBJdGVyYXRvcltEaWN0W3N0ciwgQW55XV06DQogICAgIiIiWWllbGQgb25lIHJvdyBwZXIgbW92ZSwgdHJlYXRpbmcgc2lkZS10by1tb3ZlIGFzIHRoZSB0YXJnZXQgcGxheWVyLiIiIg0KICAgIHdpdGggcGduX3BhdGgub3BlbigiciIsIGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpIGFzIGhhbmRsZToNCiAgICAgICAgZ2FtZV9pbmRleCA9IDANCg0KICAgICAgICB3aGlsZSBUcnVlOg0KICAgICAgICAgICAgZ2FtZSA9IGNoZXNzLnBnbi5yZWFkX2dhbWUoaGFuZGxlKQ0KICAgICAgICAgICAgaWYgZ2FtZSBpcyBOb25lOg0KICAgICAgICAgICAgICAgIGJyZWFrDQoNCiAgICAgICAgICAgIGdhbWVfaW5kZXggKz0gMQ0KICAgICAgICAgICAgaGVhZGVycyA9IGdhbWUuaGVhZGVycw0KICAgICAgICAgICAgaWYgaGVhZGVycy5nZXQoIlNldFVwIikgPT0gIjEiOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQoNCiAgICAgICAgICAgIHdoaXRlX25hbWUgPSBoZWFkZXJzLmdldCgiV2hpdGUiLCAiV2hpdGUiKQ0KICAgICAgICAgICAgYmxhY2tfbmFtZSA9IGhlYWRlcnMuZ2V0KCJCbGFjayIsICJCbGFjayIpDQogICAgICAgICAgICB3aGl0ZV9pZCA9IHBsYXllcl9pZCh3aGl0ZV9uYW1lKQ0KICAgICAgICAgICAgYmxhY2tfaWQgPSBwbGF5ZXJfaWQoYmxhY2tfbmFtZSkNCg0KICAgICAgICAgICAgaGlzdG9yeV93aGl0ZTogRGVxdWVbRGljdFtzdHIsIEFueV1dID0gZGVxdWUobWF4bGVuPWhpc3RvcnlfcGxpZXMpDQogICAgICAgICAgICBoaXN0b3J5X2JsYWNrOiBEZXF1ZVtEaWN0W3N0ciwgQW55XV0gPSBkZXF1ZShtYXhsZW49aGlzdG9yeV9wbGllcykNCg0KICAgICAgICAgICAgYm9hcmQgPSBnYW1lLmJvYXJkKCkNCiAgICAgICAgICAgIHBpZWNlX2lkX2J5X3NxdWFyZSwgcHJvbW90aW9uX2NvdW50ZXJzID0gaW5pdGlhbGl6ZV9waWVjZV9pZGVudGl0eV90cmFja2VyKGJvYXJkKQ0KICAgICAgICAgICAgZ2FtZV9pZF9iYXNlID0gaGVhZGVycy5nZXQoIkdhbWVJZCIpIG9yIGYiZ2FtZV97Z2FtZV9pbmRleH0iDQogICAgICAgICAgICBnYW1lX2lkID0gZiJ7cGduX3BhdGguc3RlbX06e2dhbWVfaWRfYmFzZX0iDQoNCiAgICAgICAgICAgIG5vZGUgPSBnYW1lCiAgICAgICAgICAgIHBseV9pbmRleCA9IDAKICAgICAgICAgICAgd2hpbGUgbm9kZS52YXJpYXRpb25zOgogICAgICAgICAgICAgICAgbmV4dF9ub2RlID0gbm9kZS52YXJpYXRpb24oMCkKICAgICAgICAgICAgICAgIG1vdmUgPSBuZXh0X25vZGUubW92ZQoKICAgICAgICAgICAgICAgIGlmIG1vdmUgPT0gY2hlc3MuTW92ZS5udWxsKCk6CiAgICAgICAgICAgICAgICAgICAgIyBTb21lIHNvdXJjZSBQR05zIGVuY29kZSBtaXNzaW5nL3Vua25vd24gcGxpZXMgYXMgIi0tIi4KICAgICAgICAgICAgICAgICAgICAjIFByZXNlcnZlIGRvd25zdHJlYW0gbW92ZSBhbGlnbm1lbnQgYnkgYWR2YW5jaW5nIHRoZSBib2FyZCB0dXJuLAogICAgICAgICAgICAgICAgICAgICMgYnV0IGRvIG5vdCBlbWl0IGEgc2FtcGxlIG9yIGluamVjdCBhIGZha2UgaGlzdG9yeSBldmVudC4KICAgICAgICAgICAgICAgICAgICBib2FyZC5wdXNoKG1vdmUpCiAgICAgICAgICAgICAgICAgICAgbm9kZSA9IG5leHRfbm9kZQogICAgICAgICAgICAgICAgICAgIHBseV9pbmRleCArPSAxCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgICAgICAgICBtb3Zlcl9pc193aGl0ZSA9IGJvb2woYm9hcmQudHVybiA9PSBjaGVzcy5XSElURSkNCiAgICAgICAgICAgICAgICBtb3ZlZF9waWVjZV9pZCA9IGN1cnJlbnRfcGllY2VfaWRlbnRpdHkocGllY2VfaWRfYnlfc3F1YXJlLCBtb3ZlLmZyb21fc3F1YXJlKQ0KICAgICAgICAgICAgICAgIG1vdmVkX3BpZWNlX3Nsb3QgPSBjYW5vbmljYWxfcGllY2Vfc2xvdChtb3ZlZF9waWVjZV9pZCkNCiAgICAgICAgICAgICAgICBwaWVjZV9zbG90X3RvX3NxdWFyZSA9IGN1cnJlbnRfb3JpZ2luYWxfcGllY2Vfc2xvdF9zcXVhcmVfbWFwKHBpZWNlX2lkX2J5X3NxdWFyZSwgYm9hcmQudHVybiwgYm9vbChtb3Zlcl9pc193aGl0ZSkpDQogICAgICAgICAgICAgICAgdGFyZ2V0X25hbWUgPSB3aGl0ZV9uYW1lIGlmIG1vdmVyX2lzX3doaXRlIGVsc2UgYmxhY2tfbmFtZQ0KICAgICAgICAgICAgICAgIHRhcmdldF9waWQgPSB3aGl0ZV9pZCBpZiBtb3Zlcl9pc193aGl0ZSBlbHNlIGJsYWNrX2lkDQogICAgICAgICAgICAgICAgdGFyZ2V0X2hpc3QgPSBoaXN0b3J5X3doaXRlIGlmIG1vdmVyX2lzX3doaXRlIGVsc2UgaGlzdG9yeV9ibGFjaw0KDQogICAgICAgICAgICAgICAgcm93ID0gew0KICAgICAgICAgICAgICAgICAgICAiZGF0YXNldF90YWciOiBzdHIoQ09ORklHWyJkYXRhc2V0X3RhZyJdKSwNCiAgICAgICAgICAgICAgICAgICAgImdhbWVfaWQiOiBnYW1lX2lkLA0KICAgICAgICAgICAgICAgICAgICAiZ2FtZV9pbmRleCI6IGdhbWVfaW5kZXgsDQogICAgICAgICAgICAgICAgICAgICJwbHlfaW5kZXgiOiBwbHlfaW5kZXgsDQogICAgICAgICAgICAgICAgICAgICJmdWxsbW92ZV9udW1iZXIiOiBib2FyZC5mdWxsbW92ZV9udW1iZXIsDQogICAgICAgICAgICAgICAgICAgICJmZW5fYmVmb3JlIjogYm9hcmQuZmVuKCksDQogICAgICAgICAgICAgICAgICAgICJwbGF5ZWRfdWNpIjogbW92ZS51Y2koKSwNCiAgICAgICAgICAgICAgICAgICAgIm1vdmVkX3BpZWNlX2lkIjogbW92ZWRfcGllY2VfaWQsDQogICAgICAgICAgICAgICAgICAgICJtb3ZlZF9waWVjZV9zbG90IjogbW92ZWRfcGllY2Vfc2xvdCBvciAiIiwNCiAgICAgICAgICAgICAgICAgICAgInBpZWNlX3Nsb3RfdG9fc3F1YXJlIjogcGllY2Vfc2xvdF90b19zcXVhcmUsDQogICAgICAgICAgICAgICAgICAgICJ0YXJnZXRfZnJvbV9zcSI6IG5vcm1hbGl6ZV9zcXVhcmUobW92ZS5mcm9tX3NxdWFyZSwgbW92ZXJfaXNfd2hpdGUpLA0KICAgICAgICAgICAgICAgICAgICAidGFyZ2V0X3RvX3NxIjogbm9ybWFsaXplX3NxdWFyZShtb3ZlLnRvX3NxdWFyZSwgbW92ZXJfaXNfd2hpdGUpLA0KICAgICAgICAgICAgICAgICAgICAidGFyZ2V0X3Byb21vdGlvbiI6IHByb21vdGlvbl9pbmRleChtb3ZlKSwNCiAgICAgICAgICAgICAgICAgICAgInRhcmdldF91c2VybmFtZSI6IHRhcmdldF9uYW1lLA0KICAgICAgICAgICAgICAgICAgICAicGxheWVyX2lkIjogdGFyZ2V0X3BpZCwNCiAgICAgICAgICAgICAgICAgICAgInRhcmdldF9pc193aGl0ZSI6IGludChtb3Zlcl9pc193aGl0ZSksDQogICAgICAgICAgICAgICAgICAgICJ3aGl0ZSI6IHdoaXRlX25hbWUsDQogICAgICAgICAgICAgICAgICAgICJibGFjayI6IGJsYWNrX25hbWUsDQogICAgICAgICAgICAgICAgICAgICJ3aGl0ZV9lbG8iOiBzYWZlX2ludChoZWFkZXJzLmdldCgiV2hpdGVFbG8iKSksDQogICAgICAgICAgICAgICAgICAgICJibGFja19lbG8iOiBzYWZlX2ludChoZWFkZXJzLmdldCgiQmxhY2tFbG8iKSksDQogICAgICAgICAgICAgICAgICAgICJlY28iOiBoZWFkZXJzLmdldCgiRUNPIiksDQogICAgICAgICAgICAgICAgICAgICJvcGVuaW5nIjogaGVhZGVycy5nZXQoIk9wZW5pbmciKSwNCiAgICAgICAgICAgICAgICAgICAgInRpbWVfY29udHJvbCI6IGhlYWRlcnMuZ2V0KCJUaW1lQ29udHJvbCIpLA0KICAgICAgICAgICAgICAgICAgICAicmVzdWx0IjogaGVhZGVycy5nZXQoIlJlc3VsdCIpLA0KICAgICAgICAgICAgICAgICAgICAicGhhc2VfY29kZSI6IHBoYXNlX2NvZGVfZnJvbV9mdWxsbW92ZShib2FyZC5mdWxsbW92ZV9udW1iZXIpLA0KICAgICAgICAgICAgICAgICAgICAiaGlzdG9yeSI6IGxpc3QodGFyZ2V0X2hpc3QpLA0KICAgICAgICAgICAgICAgIH0NCiAgICAgICAgICAgICAgICB5aWVsZCByb3cNCg0KICAgICAgICAgICAgICAgIGhpc3Rvcnlfd2hpdGUuYXBwZW5kKGJ1aWxkX2hpc3RvcnlfZW50cnkoYm9hcmQsIG1vdmUsIGNoZXNzLldISVRFLCBUcnVlLCBtb3ZlZF9waWVjZV9pZD1tb3ZlZF9waWVjZV9pZCkpDQogICAgICAgICAgICAgICAgaGlzdG9yeV9ibGFjay5hcHBlbmQoYnVpbGRfaGlzdG9yeV9lbnRyeShib2FyZCwgbW92ZSwgY2hlc3MuQkxBQ0ssIEZhbHNlLCBtb3ZlZF9waWVjZV9pZD1tb3ZlZF9waWVjZV9pZCkpDQoNCiAgICAgICAgICAgICAgICBhcHBseV9waWVjZV9pZGVudGl0eV9tb3ZlKGJvYXJkLCBwaWVjZV9pZF9ieV9zcXVhcmUsIG1vdmUsIHByb21vdGlvbl9jb3VudGVycykNCiAgICAgICAgICAgICAgICBib2FyZC5wdXNoKG1vdmUpDQogICAgICAgICAgICAgICAgbm9kZSA9IG5leHRfbm9kZQ0KICAgICAgICAgICAgICAgIHBseV9pbmRleCArPSAxDQoNCg0KZGVmIG1haW4oKSAtPiBOb25lOg0KICAgICIiIlJ1biBtdWx0aS1wbGF5ZXIgcGFyc2luZyBmcm9tIG1lcmdlZCBwcmV0cmFpbmluZyBQR04uIiIiDQogICAgaW5wdXRfcGduID0gUGF0aChDT05GSUdbImlucHV0X3BnbiJdKQ0KICAgIGlmIG5vdCBpbnB1dF9wZ24uZXhpc3RzKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTWVyZ2VkIHByZXRyYWluIFBHTiBub3QgZm91bmQ6IHtpbnB1dF9wZ259IikNCg0KICAgIG91dF9wYXRoID0gUGF0aChDT05GSUdbIm91dHB1dF9qc29ubCJdKQ0KICAgIG91dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQoNCiAgICByb3dzID0gMA0KICAgIGdhbWVzID0gMA0KICAgIHByb2dyZXNzX2V2ZXJ5ID0gaW50KENPTkZJR1sicHJvZ3Jlc3NfZXZlcnlfZ2FtZXMiXSkNCg0KICAgIHdpdGggb3V0X3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIG91dDoNCiAgICAgICAgZm9yIHJvdyBpbiBpdGVyX2FsbF9wbGF5ZXJfcG9zaXRpb25zKGlucHV0X3BnbiwgaW50KENPTkZJR1siaGlzdG9yeV9wbGllcyJdKSk6DQogICAgICAgICAgICBvdXQud3JpdGUoanNvbi5kdW1wcyhyb3csIGVuc3VyZV9hc2NpaT1GYWxzZSkgKyAiXG4iKQ0KICAgICAgICAgICAgcm93cyArPSAxDQogICAgICAgICAgICBpZiBpbnQocm93WyJwbHlfaW5kZXgiXSkgPT0gMDoNCiAgICAgICAgICAgICAgICBnYW1lcyArPSAxDQogICAgICAgICAgICAgICAgaWYgYm9vbChDT05GSUcuZ2V0KCJ2ZXJib3NlIiwgVHJ1ZSkpIGFuZCBwcm9ncmVzc19ldmVyeSA+IDAgYW5kIGdhbWVzICUgcHJvZ3Jlc3NfZXZlcnkgPT0gMDoNCiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiJQcm9jZXNzZWQgZ2FtZXM9e2dhbWVzfSwgcm93cz17cm93c30iKQ0KDQogICAgaWYgYm9vbChDT05GSUcuZ2V0KCJ2ZXJib3NlIiwgVHJ1ZSkpOg0KICAgICAgICBwcmludChmIlNhdmVkIHJvd3M9e3Jvd3N9IGZyb20gZ2FtZXM9e2dhbWVzfSAtPiB7b3V0X3BhdGh9IikNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0KDQoNCg0KDQoNCg==",
  "scripts/script1_parse_pgn_to_positions.py": "77u/IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiUGFyc2UgUEdOIGFuZCBidWlsZCB0YXJnZXQtbW92ZSBzYW1wbGVzIHdpdGggc3RydWN0dXJlZCBoaXN0b3J5LiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBqc29uDQppbXBvcnQgcmUNCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlcXVlDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIERlcXVlLCBEaWN0LCBJdGVyYXRvciwgTGlzdCwgT3B0aW9uYWwsIFR1cGxlDQoNCmltcG9ydCBjaGVzcw0KaW1wb3J0IGNoZXNzLnBnbg0KDQpmcm9tIGNoZXNzX2ZlYXR1cmVfdXRpbHMgaW1wb3J0ICgNCiAgICBhcHBseV9waWVjZV9pZGVudGl0eV9tb3ZlLA0KICAgIGJ1aWxkX2hpc3RvcnlfZW50cnksDQogICAgY2Fub25pY2FsX3BpZWNlX3Nsb3QsDQogICAgY3VycmVudF9vcmlnaW5hbF9waWVjZV9zbG90X3NxdWFyZV9tYXAsDQogICAgY3VycmVudF9waWVjZV9pZGVudGl0eSwNCiAgICBpbml0aWFsaXplX3BpZWNlX2lkZW50aXR5X3RyYWNrZXIsDQogICAgbm9ybWFsaXplX3NxdWFyZSwNCiAgICBwaGFzZV9jb2RlX2Zyb21fZnVsbG1vdmUsDQopDQpmcm9tIHBpcGVsaW5lX2NvbmZpZyBpbXBvcnQgKA0KICAgIERBVEFTRVRfVEFHLA0KICAgIEhJU1RPUllfUExJRVMsDQogICAgVEFSR0VUX05BTUVfQUxJQVNFUywNCiAgICBUQVJHRVRfVVNFUk5BTUUsDQogICAgcHJvY2Vzc2VkX2RpciwNCiAgICByYXdfcGduLA0KKQ0KDQoNCkNPTkZJRyA9IHsNCiAgICAiZGF0YXNldF90YWciOiBEQVRBU0VUX1RBRywNCiAgICAicGduX3BhdGgiOiByYXdfcGduKERBVEFTRVRfVEFHKSwNCiAgICAidGFyZ2V0X3VzZXJuYW1lIjogVEFSR0VUX1VTRVJOQU1FLA0KICAgICJ0YXJnZXRfbmFtZV9hbGlhc2VzIjogVEFSR0VUX05BTUVfQUxJQVNFUywNCiAgICAiaGlzdG9yeV9wbGllcyI6IEhJU1RPUllfUExJRVMsDQogICAgIm91dHB1dF9qc29ubCI6IHByb2Nlc3NlZF9kaXIoREFUQVNFVF9UQUcpIC8gInBvc2l0aW9uc19oaXN0b3J5Lmpzb25sIiwNCiAgICAidmVyYm9zZSI6IFRydWUsDQp9DQoNCkNMS19QQVRURVJOID0gcmUuY29tcGlsZShyIlxbJWNsa1xzKyhbMC05XSspOihbMC05XXsyfSk6KFswLTldezJ9KVxdIikNCk5PTl9BTE5VTV9SRSA9IHJlLmNvbXBpbGUociJbXmEtejAtOV0rIikNClBBUkVOU19SRSA9IHJlLmNvbXBpbGUociJcKFteKV0qXCkiKQ0KDQoNCmRlZiBwYXJzZV9jbG9ja190b19zZWNvbmRzKGNvbW1lbnQ6IHN0cikgLT4gT3B0aW9uYWxbaW50XToNCiAgICAiIiJSZXR1cm4gY2xvY2sgc2Vjb25kcyBmcm9tIGEgTGljaGVzcy1zdHlsZSBbJWNsayBoOm1tOnNzXSBjb21tZW50LiIiIg0KICAgIGlmIG5vdCBjb21tZW50Og0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIG1hdGNoID0gQ0xLX1BBVFRFUk4uc2VhcmNoKGNvbW1lbnQpDQogICAgaWYgbm90IG1hdGNoOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHJldHVybiBpbnQobWF0Y2guZ3JvdXAoMSkpICogMzYwMCArIGludChtYXRjaC5ncm91cCgyKSkgKiA2MCArIGludChtYXRjaC5ncm91cCgzKSkNCg0KDQpkZWYgc2FmZV9pbnQoeDogT3B0aW9uYWxbc3RyXSkgLT4gT3B0aW9uYWxbaW50XToNCiAgICAiIiJDb252ZXJ0IGhlYWRlciB0ZXh0IHRvIGludCB3aGVuIHBvc3NpYmxlLiIiIg0KICAgIGlmIHggaXMgTm9uZToNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICB4ID0geC5zdHJpcCgpDQogICAgaWYgbm90IHggb3IgeCA9PSAiPyI6DQogICAgICAgIHJldHVybiBOb25lDQogICAgdHJ5Og0KICAgICAgICByZXR1cm4gaW50KHgpDQogICAgZXhjZXB0IFZhbHVlRXJyb3I6DQogICAgICAgIHJldHVybiBOb25lDQoNCg0KZGVmIGNsZWFuX25hbWUobmFtZTogc3RyKSAtPiBzdHI6DQogICAgIiIiTm9ybWFsaXplIHBsYXllciBuYW1lcyBmb3Igcm9idXN0IGZ1enp5IG1hdGNoaW5nLiIiIg0KICAgIHRleHQgPSBQQVJFTlNfUkUuc3ViKCIgIiwgKG5hbWUgb3IgIiIpLmxvd2VyKCkpDQogICAgdGV4dCA9IE5PTl9BTE5VTV9SRS5zdWIoIiAiLCB0ZXh0KQ0KICAgIHJldHVybiAiICIuam9pbih0ZXh0LnNwbGl0KCkpDQoNCg0KZGVmIG5hbWVfdG9rZW5zKG5hbWU6IHN0cikgLT4gTGlzdFtzdHJdOg0KICAgICIiIlJldHVybiBjbGVhbmVkIG5vbi1lbXB0eSB0b2tlbnMgZnJvbSBwbGF5ZXIgbmFtZS4iIiINCiAgICByZXR1cm4gW3QgZm9yIHQgaW4gY2xlYW5fbmFtZShuYW1lKS5zcGxpdCgpIGlmIHRdDQoNCg0KZGVmIGJ1aWxkX3RhcmdldF9uYW1lX3Byb2ZpbGVzKHRhcmdldF91c2VybmFtZTogc3RyLCBhbGlhc2VzOiBMaXN0W3N0cl0pIC0+IExpc3RbRGljdFtzdHIsIEFueV1dOg0KICAgICIiIkJ1aWxkIG5vcm1hbGl6ZWQgdGFyZ2V0IG5hbWUgcHJvZmlsZXMgZnJvbSBjYW5vbmljYWwgbmFtZSBhbmQgYWxpYXNlcy4iIiINCiAgICBuYW1lcyA9IFt0YXJnZXRfdXNlcm5hbWVdICsgW2EgZm9yIGEgaW4gYWxpYXNlcyBpZiBzdHIoYSkuc3RyaXAoKV0NCiAgICBwcm9maWxlczogTGlzdFtEaWN0W3N0ciwgQW55XV0gPSBbXQ0KDQogICAgc2VlbiA9IHNldCgpDQogICAgZm9yIHJhdyBpbiBuYW1lczoNCiAgICAgICAgY2xlYW5lZCA9IGNsZWFuX25hbWUocmF3KQ0KICAgICAgICBpZiBub3QgY2xlYW5lZCBvciBjbGVhbmVkIGluIHNlZW46DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBzZWVuLmFkZChjbGVhbmVkKQ0KICAgICAgICB0b2tlbnMgPSBuYW1lX3Rva2VucyhyYXcpDQogICAgICAgIGlmIG5vdCB0b2tlbnM6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBwcm9maWxlcy5hcHBlbmQoDQogICAgICAgICAgICB7DQogICAgICAgICAgICAgICAgInJhdyI6IHJhdywNCiAgICAgICAgICAgICAgICAiY2xlYW5lZCI6IGNsZWFuZWQsDQogICAgICAgICAgICAgICAgInRva2VucyI6IHNldCh0b2tlbnMpLA0KICAgICAgICAgICAgICAgICJzdXJuYW1lIjogdG9rZW5zWy0xXSwNCiAgICAgICAgICAgICAgICAiZmlyc3RfaW5pdGlhbCI6IHRva2Vuc1swXVswXSwNCiAgICAgICAgICAgIH0NCiAgICAgICAgKQ0KDQogICAgaWYgbm90IHByb2ZpbGVzOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJObyB2YWxpZCB0YXJnZXQgbmFtZSBwcm9maWxlIGNvdWxkIGJlIGJ1aWx0IikNCiAgICByZXR1cm4gcHJvZmlsZXMNCg0KDQpkZWYgbWF0Y2hfc2NvcmUoY2FuZGlkYXRlX25hbWU6IHN0ciwgcHJvZmlsZTogRGljdFtzdHIsIEFueV0pIC0+IGludDoNCiAgICAiIiJSZXR1cm4gZnV6enkgbWF0Y2ggc2NvcmUgYmV0d2VlbiBjYW5kaWRhdGUgaGVhZGVyIG5hbWUgYW5kIG9uZSB0YXJnZXQgcHJvZmlsZS4iIiINCiAgICBjYW5kaWRhdGVfY2xlYW4gPSBjbGVhbl9uYW1lKGNhbmRpZGF0ZV9uYW1lKQ0KICAgIGlmIG5vdCBjYW5kaWRhdGVfY2xlYW46DQogICAgICAgIHJldHVybiAwDQoNCiAgICBpZiBjYW5kaWRhdGVfY2xlYW4gPT0gcHJvZmlsZVsiY2xlYW5lZCJdOg0KICAgICAgICByZXR1cm4gMTAwDQoNCiAgICBpZiBjYW5kaWRhdGVfY2xlYW4gaW4gcHJvZmlsZVsiY2xlYW5lZCJdIG9yIHByb2ZpbGVbImNsZWFuZWQiXSBpbiBjYW5kaWRhdGVfY2xlYW46DQogICAgICAgIGlmIGxlbihjYW5kaWRhdGVfY2xlYW4pID49IDQ6DQogICAgICAgICAgICByZXR1cm4gOTANCg0KICAgIGNhbmRfdG9rZW5zID0gc2V0KG5hbWVfdG9rZW5zKGNhbmRpZGF0ZV9uYW1lKSkNCiAgICBvdmVybGFwID0gbGVuKGNhbmRfdG9rZW5zICYgcHJvZmlsZVsidG9rZW5zIl0pDQoNCiAgICBpZiBvdmVybGFwID49IDI6DQogICAgICAgIHJldHVybiA4MA0KDQogICAgaWYgcHJvZmlsZVsic3VybmFtZSJdIGluIGNhbmRfdG9rZW5zOg0KICAgICAgICBjYW5kX2ZpcnN0X2luaXRpYWwgPSAiIg0KICAgICAgICBjYW5kX2xpc3QgPSBuYW1lX3Rva2VucyhjYW5kaWRhdGVfbmFtZSkNCiAgICAgICAgaWYgY2FuZF9saXN0Og0KICAgICAgICAgICAgY2FuZF9maXJzdF9pbml0aWFsID0gY2FuZF9saXN0WzBdWzBdDQogICAgICAgIGlmIG92ZXJsYXAgPj0gMSBhbmQgY2FuZF9maXJzdF9pbml0aWFsID09IHByb2ZpbGVbImZpcnN0X2luaXRpYWwiXToNCiAgICAgICAgICAgIHJldHVybiA3NQ0KICAgICAgICBpZiBvdmVybGFwID49IDE6DQogICAgICAgICAgICByZXR1cm4gNzANCg0KICAgIHJldHVybiAwDQoNCg0KZGVmIG1hdGNoX3RhcmdldF9jb2xvcih3aGl0ZV9uYW1lOiBzdHIsIGJsYWNrX25hbWU6IHN0ciwgcHJvZmlsZXM6IExpc3RbRGljdFtzdHIsIEFueV1dKSAtPiBPcHRpb25hbFtjaGVzcy5Db2xvcl06DQogICAgIiIiSW5mZXIgd2hldGhlciB0YXJnZXQgaXMgd2hpdGUvYmxhY2sgdXNpbmcgcm9idXN0IG5hbWUgbWF0Y2hpbmcuIiIiDQogICAgd2hpdGVfYmVzdCA9IG1heChtYXRjaF9zY29yZSh3aGl0ZV9uYW1lLCBwKSBmb3IgcCBpbiBwcm9maWxlcykNCiAgICBibGFja19iZXN0ID0gbWF4KG1hdGNoX3Njb3JlKGJsYWNrX25hbWUsIHApIGZvciBwIGluIHByb2ZpbGVzKQ0KDQogICAgdGhyZXNob2xkID0gNzANCiAgICB3aGl0ZV9oaXQgPSB3aGl0ZV9iZXN0ID49IHRocmVzaG9sZA0KICAgIGJsYWNrX2hpdCA9IGJsYWNrX2Jlc3QgPj0gdGhyZXNob2xkDQoNCiAgICBpZiB3aGl0ZV9oaXQgYW5kIG5vdCBibGFja19oaXQ6DQogICAgICAgIHJldHVybiBjaGVzcy5XSElURQ0KICAgIGlmIGJsYWNrX2hpdCBhbmQgbm90IHdoaXRlX2hpdDoNCiAgICAgICAgcmV0dXJuIGNoZXNzLkJMQUNLDQogICAgaWYgd2hpdGVfaGl0IGFuZCBibGFja19oaXQ6DQogICAgICAgIHJldHVybiBjaGVzcy5XSElURSBpZiB3aGl0ZV9iZXN0ID49IGJsYWNrX2Jlc3QgZWxzZSBjaGVzcy5CTEFDSw0KICAgIHJldHVybiBOb25lDQoNCg0KZGVmIHByb21vdGlvbl9pbmRleChtb3ZlOiBjaGVzcy5Nb3ZlKSAtPiBpbnQ6DQogICAgIiIiTWFwIHByb21vdGlvbiBwaWVjZSB0eXBlIHRvIGNvbXBhY3QgaWQgKDAgbm9uZSwgMSBOLCAyIEIsIDMgUiwgNCBRKS4iIiINCiAgICBpZiBtb3ZlLnByb21vdGlvbiBpcyBOb25lOg0KICAgICAgICByZXR1cm4gMA0KICAgIG1hcHBpbmcgPSB7Y2hlc3MuS05JR0hUOiAxLCBjaGVzcy5CSVNIT1A6IDIsIGNoZXNzLlJPT0s6IDMsIGNoZXNzLlFVRUVOOiA0fQ0KICAgIHJldHVybiBpbnQobWFwcGluZy5nZXQobW92ZS5wcm9tb3Rpb24sIDApKQ0KDQoNCmRlZiBpdGVyX3RhcmdldF9wbGF5ZXJfcG9zaXRpb25zKHBnbl9wYXRoOiBQYXRoLCB0YXJnZXRfdXNlcm5hbWU6IHN0ciwgYWxpYXNlczogTGlzdFtzdHJdLCBoaXN0b3J5X3BsaWVzOiBpbnQpIC0+IEl0ZXJhdG9yW0RpY3Rbc3RyLCBBbnldXToNCiAgICAiIiJZaWVsZCBvbmUgcm93IHBlciB0YXJnZXQgbW92ZSB3aXRoIHRyYWlsaW5nIGhpc3RvcnkgZnJvbSByZWNlbnQgcGxpZXMuIiIiDQogICAgdGFyZ2V0X3Byb2ZpbGVzID0gYnVpbGRfdGFyZ2V0X25hbWVfcHJvZmlsZXModGFyZ2V0X3VzZXJuYW1lLCBhbGlhc2VzKQ0KDQogICAgd2l0aCBwZ25fcGF0aC5vcGVuKCJyIiwgZW5jb2Rpbmc9InV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikgYXMgaGFuZGxlOg0KICAgICAgICBnYW1lX2luZGV4ID0gMA0KDQogICAgICAgIHdoaWxlIFRydWU6DQogICAgICAgICAgICBnYW1lID0gY2hlc3MucGduLnJlYWRfZ2FtZShoYW5kbGUpDQogICAgICAgICAgICBpZiBnYW1lIGlzIE5vbmU6DQogICAgICAgICAgICAgICAgYnJlYWsNCg0KICAgICAgICAgICAgZ2FtZV9pbmRleCArPSAxDQogICAgICAgICAgICBoZWFkZXJzID0gZ2FtZS5oZWFkZXJzDQoNCiAgICAgICAgICAgIHdoaXRlID0gaGVhZGVycy5nZXQoIldoaXRlIiwgIiIpDQogICAgICAgICAgICBibGFjayA9IGhlYWRlcnMuZ2V0KCJCbGFjayIsICIiKQ0KDQogICAgICAgICAgICB0YXJnZXRfY29sb3IgPSBtYXRjaF90YXJnZXRfY29sb3Iod2hpdGUsIGJsYWNrLCB0YXJnZXRfcHJvZmlsZXMpDQogICAgICAgICAgICBpZiB0YXJnZXRfY29sb3IgaXMgTm9uZToNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KDQogICAgICAgICAgICB0YXJnZXRfaXNfd2hpdGUgPSBib29sKHRhcmdldF9jb2xvciA9PSBjaGVzcy5XSElURSkNCiAgICAgICAgICAgIGhpc3Rvcnk6IERlcXVlW0RpY3Rbc3RyLCBBbnldXSA9IGRlcXVlKG1heGxlbj1oaXN0b3J5X3BsaWVzKQ0KDQogICAgICAgICAgICBib2FyZCA9IGdhbWUuYm9hcmQoKQ0KICAgICAgICAgICAgcGllY2VfaWRfYnlfc3F1YXJlLCBwcm9tb3Rpb25fY291bnRlcnMgPSBpbml0aWFsaXplX3BpZWNlX2lkZW50aXR5X3RyYWNrZXIoYm9hcmQpDQogICAgICAgICAgICBnYW1lX2lkID0gaGVhZGVycy5nZXQoIkdhbWVJZCIpIG9yIGYiZ2FtZV97Z2FtZV9pbmRleH0iDQoNCiAgICAgICAgICAgIG5vZGUgPSBnYW1lDQogICAgICAgICAgICBwbHlfaW5kZXggPSAwDQogICAgICAgICAgICB3aGlsZSBub2RlLnZhcmlhdGlvbnM6DQogICAgICAgICAgICAgICAgbmV4dF9ub2RlID0gbm9kZS52YXJpYXRpb24oMCkNCiAgICAgICAgICAgICAgICBtb3ZlID0gbmV4dF9ub2RlLm1vdmUNCg0KICAgICAgICAgICAgICAgIGlmIG1vdmUgPT0gY2hlc3MuTW92ZS5udWxsKCk6DQogICAgICAgICAgICAgICAgICAgICMgU29tZSBzb3VyY2UgUEdOcyBlbmNvZGUgbWlzc2luZy91bmtub3duIHBsaWVzIGFzICItLSIuDQogICAgICAgICAgICAgICAgICAgICMgUHJlc2VydmUgZG93bnN0cmVhbSBtb3ZlIGFsaWdubWVudCBieSBhZHZhbmNpbmcgdGhlIGJvYXJkIHR1cm4sDQogICAgICAgICAgICAgICAgICAgICMgYnV0IGRvIG5vdCBlbWl0IGEgc2FtcGxlIG9yIGluamVjdCBhIGZha2UgaGlzdG9yeSBldmVudC4NCiAgICAgICAgICAgICAgICAgICAgYm9hcmQucHVzaChtb3ZlKQ0KICAgICAgICAgICAgICAgICAgICBub2RlID0gbmV4dF9ub2RlDQogICAgICAgICAgICAgICAgICAgIHBseV9pbmRleCArPSAxDQogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlDQoNCiAgICAgICAgICAgICAgICBtb3ZlZF9waWVjZV9pZCA9IGN1cnJlbnRfcGllY2VfaWRlbnRpdHkocGllY2VfaWRfYnlfc3F1YXJlLCBtb3ZlLmZyb21fc3F1YXJlKQ0KICAgICAgICAgICAgICAgIG1vdmVkX3BpZWNlX3Nsb3QgPSBjYW5vbmljYWxfcGllY2Vfc2xvdChtb3ZlZF9waWVjZV9pZCkNCiAgICAgICAgICAgICAgICBwaWVjZV9zbG90X3RvX3NxdWFyZSA9IGN1cnJlbnRfb3JpZ2luYWxfcGllY2Vfc2xvdF9zcXVhcmVfbWFwKHBpZWNlX2lkX2J5X3NxdWFyZSwgdGFyZ2V0X2NvbG9yLCBib29sKHRhcmdldF9pc193aGl0ZSkpDQoNCiAgICAgICAgICAgICAgICBpZiBib2FyZC50dXJuID09IHRhcmdldF9jb2xvcjoNCiAgICAgICAgICAgICAgICAgICAgcm93ID0gew0KICAgICAgICAgICAgICAgICAgICAgICAgImRhdGFzZXRfdGFnIjogREFUQVNFVF9UQUcsDQogICAgICAgICAgICAgICAgICAgICAgICAiZ2FtZV9pZCI6IGdhbWVfaWQsDQogICAgICAgICAgICAgICAgICAgICAgICAiZ2FtZV9pbmRleCI6IGdhbWVfaW5kZXgsDQogICAgICAgICAgICAgICAgICAgICAgICAicGx5X2luZGV4IjogcGx5X2luZGV4LA0KICAgICAgICAgICAgICAgICAgICAgICAgImZ1bGxtb3ZlX251bWJlciI6IGJvYXJkLmZ1bGxtb3ZlX251bWJlciwNCiAgICAgICAgICAgICAgICAgICAgICAgICJmZW5fYmVmb3JlIjogYm9hcmQuZmVuKCksDQogICAgICAgICAgICAgICAgICAgICAgICAicGxheWVkX3VjaSI6IG1vdmUudWNpKCksDQogICAgICAgICAgICAgICAgICAgICAgICAibW92ZWRfcGllY2VfaWQiOiBtb3ZlZF9waWVjZV9pZCwNCiAgICAgICAgICAgICAgICAgICAgICAgICJtb3ZlZF9waWVjZV9zbG90IjogbW92ZWRfcGllY2Vfc2xvdCBvciAiIiwNCiAgICAgICAgICAgICAgICAgICAgICAgICJwaWVjZV9zbG90X3RvX3NxdWFyZSI6IHBpZWNlX3Nsb3RfdG9fc3F1YXJlLA0KICAgICAgICAgICAgICAgICAgICAgICAgInRhcmdldF9mcm9tX3NxIjogbm9ybWFsaXplX3NxdWFyZShtb3ZlLmZyb21fc3F1YXJlLCB0YXJnZXRfaXNfd2hpdGUpLA0KICAgICAgICAgICAgICAgICAgICAgICAgInRhcmdldF90b19zcSI6IG5vcm1hbGl6ZV9zcXVhcmUobW92ZS50b19zcXVhcmUsIHRhcmdldF9pc193aGl0ZSksDQogICAgICAgICAgICAgICAgICAgICAgICAidGFyZ2V0X3Byb21vdGlvbiI6IHByb21vdGlvbl9pbmRleChtb3ZlKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICJ0YXJnZXRfdXNlcm5hbWUiOiB0YXJnZXRfdXNlcm5hbWUsDQogICAgICAgICAgICAgICAgICAgICAgICAidGFyZ2V0X2lzX3doaXRlIjogaW50KHRhcmdldF9pc193aGl0ZSksDQogICAgICAgICAgICAgICAgICAgICAgICAid2hpdGUiOiB3aGl0ZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICJibGFjayI6IGJsYWNrLA0KICAgICAgICAgICAgICAgICAgICAgICAgIndoaXRlX2VsbyI6IHNhZmVfaW50KGhlYWRlcnMuZ2V0KCJXaGl0ZUVsbyIpKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICJibGFja19lbG8iOiBzYWZlX2ludChoZWFkZXJzLmdldCgiQmxhY2tFbG8iKSksDQogICAgICAgICAgICAgICAgICAgICAgICAiZWNvIjogaGVhZGVycy5nZXQoIkVDTyIpLA0KICAgICAgICAgICAgICAgICAgICAgICAgIm9wZW5pbmciOiBoZWFkZXJzLmdldCgiT3BlbmluZyIpLA0KICAgICAgICAgICAgICAgICAgICAgICAgInRpbWVfY29udHJvbCI6IGhlYWRlcnMuZ2V0KCJUaW1lQ29udHJvbCIpLA0KICAgICAgICAgICAgICAgICAgICAgICAgInJlc3VsdCI6IGhlYWRlcnMuZ2V0KCJSZXN1bHQiKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICJwaGFzZV9jb2RlIjogcGhhc2VfY29kZV9mcm9tX2Z1bGxtb3ZlKGJvYXJkLmZ1bGxtb3ZlX251bWJlciksDQogICAgICAgICAgICAgICAgICAgICAgICAiaGlzdG9yeSI6IGxpc3QoaGlzdG9yeSksDQogICAgICAgICAgICAgICAgICAgIH0NCiAgICAgICAgICAgICAgICAgICAgeWllbGQgcm93DQoNCiAgICAgICAgICAgICAgICBoaXN0b3J5LmFwcGVuZChidWlsZF9oaXN0b3J5X2VudHJ5KGJvYXJkLCBtb3ZlLCB0YXJnZXRfY29sb3IsIHRhcmdldF9pc193aGl0ZSwgbW92ZWRfcGllY2VfaWQ9bW92ZWRfcGllY2VfaWQpKQ0KICAgICAgICAgICAgICAgIGFwcGx5X3BpZWNlX2lkZW50aXR5X21vdmUoYm9hcmQsIHBpZWNlX2lkX2J5X3NxdWFyZSwgbW92ZSwgcHJvbW90aW9uX2NvdW50ZXJzKQ0KICAgICAgICAgICAgICAgIGJvYXJkLnB1c2gobW92ZSkNCiAgICAgICAgICAgICAgICBub2RlID0gbmV4dF9ub2RlDQogICAgICAgICAgICAgICAgcGx5X2luZGV4ICs9IDENCg0KDQpkZWYgdmFsaWRhdGVfY29uZmlnKGNvbmZpZzogRGljdFtzdHIsIEFueV0pIC0+IE5vbmU6DQogICAgIiIiVmFsaWRhdGUgcmVxdWlyZWQgaW5wdXRzIGFuZCBjb25zdHJhaW50cyBiZWZvcmUgcGFyc2luZy4iIiINCiAgICBwZ25fcGF0aCA9IFBhdGgoY29uZmlnWyJwZ25fcGF0aCJdKQ0KICAgIGlmIG5vdCBwZ25fcGF0aC5leGlzdHMoKToNCiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJQR04gZmlsZSBub3QgZm91bmQ6IHtwZ25fcGF0aH0iKQ0KICAgIGlmIGludChjb25maWdbImhpc3RvcnlfcGxpZXMiXSkgPD0gMDoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaGlzdG9yeV9wbGllcyBtdXN0IGJlIHBvc2l0aXZlIikNCiAgICBpZiBub3Qgc3RyKGNvbmZpZ1sidGFyZ2V0X3VzZXJuYW1lIl0pLnN0cmlwKCk6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRhcmdldF91c2VybmFtZSBjYW5ub3QgYmUgZW1wdHkiKQ0KDQoNCmRlZiBtYWluKCkgLT4gTm9uZToNCiAgICAiIiJSdW4gc3RhZ2UgMSBhbmQgc2F2ZSBoaXN0b3J5LWF3YXJlIHBvc2l0aW9uIHNhbXBsZXMgYXMgSlNPTkwuIiIiDQogICAgdmFsaWRhdGVfY29uZmlnKENPTkZJRykNCg0KICAgIG91dF9wYXRoID0gUGF0aChDT05GSUdbIm91dHB1dF9qc29ubCJdKQ0KICAgIG91dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQoNCiAgICBuX3Jvd3MgPSAwDQogICAgd2l0aCBvdXRfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgb3V0Og0KICAgICAgICBmb3Igcm93IGluIGl0ZXJfdGFyZ2V0X3BsYXllcl9wb3NpdGlvbnMoDQogICAgICAgICAgICBwZ25fcGF0aD1QYXRoKENPTkZJR1sicGduX3BhdGgiXSksDQogICAgICAgICAgICB0YXJnZXRfdXNlcm5hbWU9c3RyKENPTkZJR1sidGFyZ2V0X3VzZXJuYW1lIl0pLA0KICAgICAgICAgICAgYWxpYXNlcz1saXN0KENPTkZJRy5nZXQoInRhcmdldF9uYW1lX2FsaWFzZXMiLCBbXSkpLA0KICAgICAgICAgICAgaGlzdG9yeV9wbGllcz1pbnQoQ09ORklHWyJoaXN0b3J5X3BsaWVzIl0pLA0KICAgICAgICApOg0KICAgICAgICAgICAgb3V0LndyaXRlKGpzb24uZHVtcHMocm93LCBlbnN1cmVfYXNjaWk9RmFsc2UpICsgIlxuIikNCiAgICAgICAgICAgIG5fcm93cyArPSAxDQoNCiAgICBpZiBib29sKENPTkZJRy5nZXQoInZlcmJvc2UiLCBUcnVlKSk6DQogICAgICAgIHByaW50KGYiU2F2ZWQge25fcm93c30gcm93cyB0byB7b3V0X3BhdGh9IikNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0KDQoNCg0KDQoNCg==",
  "scripts/script2_encode_policy_samples.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiRW5jb2RlIGhpc3RvcnktYXdhcmUgcG9zaXRpb24gcm93cyBpbnRvIG1vZGVsLXJlYWR5IEpTT05MIHNhbXBsZXMuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGpzb24NCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgRGljdCwgSXRlcmFibGUsIExpc3QNCg0KaW1wb3J0IGNoZXNzDQoNCmZyb20gY2hlc3NfZmVhdHVyZV91dGlscyBpbXBvcnQgT1JJR0lOQUxfUElFQ0VfU0xPVFMsIE9SSUdJTkFMX1NMT1RfVE9fSU5ERVgsIGRlbnNlX3N0YXRlX3ZlY3RvciwgaXNfdW5kZXJfaW1tZWRpYXRlX3RocmVhdA0KZnJvbSBwaXBlbGluZV9jb25maWcgaW1wb3J0IERBVEFTRVRfVEFHLCBISVNUT1JZX1BMSUVTLCBwcm9jZXNzZWRfZGlyDQoNCg0KQ09ORklHID0gew0KICAgICJkYXRhc2V0X3RhZyI6IERBVEFTRVRfVEFHLA0KICAgICJoaXN0b3J5X3BsaWVzIjogSElTVE9SWV9QTElFUywNCiAgICAiaW5wdXRfanNvbmwiOiBwcm9jZXNzZWRfZGlyKERBVEFTRVRfVEFHKSAvICJwb3NpdGlvbnNfaGlzdG9yeS5qc29ubCIsDQogICAgIm91dHB1dF9qc29ubCI6IHByb2Nlc3NlZF9kaXIoREFUQVNFVF9UQUcpIC8gInBvbGljeV9zYW1wbGVzLmpzb25sIiwNCiAgICAicHJvZ3Jlc3NfZXZlcnkiOiA1MDAwLA0KICAgICJ2ZXJib3NlIjogVHJ1ZSwNCn0NCg0KTlVNX1NRVUFSRVMgPSA2NA0KTlVNX05PTl9LSU5HX1BMQU5FUyA9IDEwDQpIQUxGS1BfQkxPQ0tfRElNID0gTlVNX1NRVUFSRVMgKiBOVU1fTk9OX0tJTkdfUExBTkVTICogTlVNX1NRVUFSRVMNCkhBTEZLUF9PV05fT0ZGU0VUID0gMA0KSEFMRktQX09QUF9PRkZTRVQgPSBIQUxGS1BfQkxPQ0tfRElNDQpDQVNUTEVfT1dOX0tfSURYID0gMiAqIEhBTEZLUF9CTE9DS19ESU0NCkNBU1RMRV9PV05fUV9JRFggPSAyICogSEFMRktQX0JMT0NLX0RJTSArIDENCkNBU1RMRV9PUFBfS19JRFggPSAyICogSEFMRktQX0JMT0NLX0RJTSArIDINCkNBU1RMRV9PUFBfUV9JRFggPSAyICogSEFMRktQX0JMT0NLX0RJTSArIDMNClRPVEFMX0ZFQVRVUkVfRElNID0gMiAqIEhBTEZLUF9CTE9DS19ESU0gKyA0DQpERU5TRV9TVEFURV9ESU0gPSAyOQ0KSElTVE9SWV9FVkVOVF9ESU0gPSAxNg0KSElTVE9SWV9ERUxUQV9ESU0gPSAxMQ0KUElFQ0VfU0xPVF9ESU0gPSBsZW4oT1JJR0lOQUxfUElFQ0VfU0xPVFMpDQoNCg0KZGVmIHBpZWNlX3Nsb3Rfc3F1YXJlX2xpc3QobWFwcGluZzogRGljdFtzdHIsIEFueV0pIC0+IExpc3RbaW50XToNCiAgICBzcXVhcmVzID0gWy0xXSAqIFBJRUNFX1NMT1RfRElNDQogICAgZm9yIHNsb3RfbmFtZSwgc3F1YXJlIGluIG1hcHBpbmcuaXRlbXMoKToNCiAgICAgICAgaWR4ID0gT1JJR0lOQUxfU0xPVF9UT19JTkRFWC5nZXQoc3RyKHNsb3RfbmFtZSkpDQogICAgICAgIGlmIGlkeCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHNxdWFyZXNbaWR4XSA9IGludChzcXVhcmUpDQogICAgcmV0dXJuIHNxdWFyZXMNCg0KDQpkZWYgbGVnYWxfcGllY2Vfc2xvdF9tYXNrKHBpZWNlX3Nsb3RfdG9fc3F1YXJlX2xpc3Q6IExpc3RbaW50XSwgbGVnYWxfbWFwOiBEaWN0W3N0ciwgTGlzdFtpbnRdXSkgLT4gTGlzdFtpbnRdOg0KICAgIG1hc2sgPSBbMF0gKiBQSUVDRV9TTE9UX0RJTQ0KICAgIGZvciBpZHgsIGZyb21fc3EgaW4gZW51bWVyYXRlKHBpZWNlX3Nsb3RfdG9fc3F1YXJlX2xpc3QpOg0KICAgICAgICBpZiBmcm9tX3NxID49IDAgYW5kIHN0cihpbnQoZnJvbV9zcSkpIGluIGxlZ2FsX21hcDoNCiAgICAgICAgICAgIG1hc2tbaWR4XSA9IDENCiAgICByZXR1cm4gbWFzaw0KDQoNCmRlZiBub3JtYWxpemVfc3F1YXJlKHNxdWFyZTogaW50LCBtb3Zlcl9pc193aGl0ZTogYm9vbCkgLT4gaW50Og0KICAgICIiIlJldHVybiBzcXVhcmUgaW5kZXggaW4gbW92ZXItcmVsYXRpdmUgY29vcmRpbmF0ZXMuIiIiDQogICAgcmV0dXJuIHNxdWFyZSBpZiBtb3Zlcl9pc193aGl0ZSBlbHNlIGNoZXNzLnNxdWFyZV9taXJyb3Ioc3F1YXJlKQ0KDQoNCmRlZiBoYWxma3Bfbm9uX2tpbmdfcGxhbmVfaWRfcmVsYXRpdmUocGllY2U6IGNoZXNzLlBpZWNlLCBtb3Zlcl9jb2xvcjogYm9vbCkgLT4gaW50IHwgTm9uZToNCiAgICAiIiJSZXR1cm4gSGFsZktQIG5vbi1raW5nIHBpZWNlIHBsYW5lIGlkIGluIG1vdmVyLXJlbGF0aXZlIHBlcnNwZWN0aXZlLiIiIg0KICAgIGlmIHBpZWNlLnBpZWNlX3R5cGUgPT0gY2hlc3MuS0lORzoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBiYXNlID0gcGllY2UucGllY2VfdHlwZSAtIDENCiAgICByZXR1cm4gYmFzZSBpZiBwaWVjZS5jb2xvciA9PSBtb3Zlcl9jb2xvciBlbHNlIDUgKyBiYXNlDQoNCg0KZGVmIGhhbGZrcF9pbmRleChraW5nX3JlbF9zcTogaW50LCBwaWVjZV9wbGFuZTogaW50LCBwaWVjZV9yZWxfc3E6IGludCwgYmxvY2tfb2Zmc2V0OiBpbnQpIC0+IGludDoNCiAgICAiIiJSZXR1cm4gZmxhdHRlbmVkIEhhbGZLUCBmZWF0dXJlIGluZGV4LiIiIg0KICAgIHJldHVybiBibG9ja19vZmZzZXQgKyAoKGtpbmdfcmVsX3NxICogTlVNX05PTl9LSU5HX1BMQU5FUyArIHBpZWNlX3BsYW5lKSAqIE5VTV9TUVVBUkVTICsgcGllY2VfcmVsX3NxKQ0KDQoNCmRlZiBlbmNvZGVfYm9hcmRfc3BhcnNlX2luZGljZXMoZmVuOiBzdHIpIC0+IExpc3RbaW50XToNCiAgICAiIiJFbmNvZGUgYm9hcmQgYXMgYWN0aXZlIEhhbGZLUCBpbmRpY2VzIHBsdXMgY2FzdGxpbmcgYml0cy4iIiINCiAgICBib2FyZCA9IGNoZXNzLkJvYXJkKGZlbikNCiAgICBhY3RpdmU6IExpc3RbaW50XSA9IFtdDQoNCiAgICBtb3Zlcl9jb2xvciA9IGJvYXJkLnR1cm4NCiAgICBvcHBfY29sb3IgPSBub3QgbW92ZXJfY29sb3INCg0KICAgIG93bl9raW5nX3NxID0gYm9hcmQua2luZyhtb3Zlcl9jb2xvcikNCiAgICBvcHBfa2luZ19zcSA9IGJvYXJkLmtpbmcob3BwX2NvbG9yKQ0KICAgIGlmIG93bl9raW5nX3NxIGlzIE5vbmUgb3Igb3BwX2tpbmdfc3EgaXMgTm9uZToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBib2FyZCB3aXRob3V0IGtpbmdzIikNCg0KICAgIG93bl9raW5nX3JlbF9zcSA9IG5vcm1hbGl6ZV9zcXVhcmUob3duX2tpbmdfc3EsIG1vdmVyX2NvbG9yKQ0KICAgIG9wcF9raW5nX3JlbF9zcSA9IG5vcm1hbGl6ZV9zcXVhcmUob3BwX2tpbmdfc3EsIG1vdmVyX2NvbG9yKQ0KDQogICAgZm9yIHNxdWFyZSwgcGllY2UgaW4gYm9hcmQucGllY2VfbWFwKCkuaXRlbXMoKToNCiAgICAgICAgcGxhbmUgPSBoYWxma3Bfbm9uX2tpbmdfcGxhbmVfaWRfcmVsYXRpdmUocGllY2UsIG1vdmVyX2NvbG9yKQ0KICAgICAgICBpZiBwbGFuZSBpcyBOb25lOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgcGllY2VfcmVsX3NxID0gbm9ybWFsaXplX3NxdWFyZShzcXVhcmUsIG1vdmVyX2NvbG9yKQ0KICAgICAgICBhY3RpdmUuYXBwZW5kKGhhbGZrcF9pbmRleChvd25fa2luZ19yZWxfc3EsIHBsYW5lLCBwaWVjZV9yZWxfc3EsIEhBTEZLUF9PV05fT0ZGU0VUKSkNCiAgICAgICAgYWN0aXZlLmFwcGVuZChoYWxma3BfaW5kZXgob3BwX2tpbmdfcmVsX3NxLCBwbGFuZSwgcGllY2VfcmVsX3NxLCBIQUxGS1BfT1BQX09GRlNFVCkpDQoNCiAgICBpZiBtb3Zlcl9jb2xvciA9PSBjaGVzcy5XSElURToNCiAgICAgICAgaWYgYm9hcmQuaGFzX2tpbmdzaWRlX2Nhc3RsaW5nX3JpZ2h0cyhjaGVzcy5XSElURSk6DQogICAgICAgICAgICBhY3RpdmUuYXBwZW5kKENBU1RMRV9PV05fS19JRFgpDQogICAgICAgIGlmIGJvYXJkLmhhc19xdWVlbnNpZGVfY2FzdGxpbmdfcmlnaHRzKGNoZXNzLldISVRFKToNCiAgICAgICAgICAgIGFjdGl2ZS5hcHBlbmQoQ0FTVExFX09XTl9RX0lEWCkNCiAgICAgICAgaWYgYm9hcmQuaGFzX2tpbmdzaWRlX2Nhc3RsaW5nX3JpZ2h0cyhjaGVzcy5CTEFDSyk6DQogICAgICAgICAgICBhY3RpdmUuYXBwZW5kKENBU1RMRV9PUFBfS19JRFgpDQogICAgICAgIGlmIGJvYXJkLmhhc19xdWVlbnNpZGVfY2FzdGxpbmdfcmlnaHRzKGNoZXNzLkJMQUNLKToNCiAgICAgICAgICAgIGFjdGl2ZS5hcHBlbmQoQ0FTVExFX09QUF9RX0lEWCkNCiAgICBlbHNlOg0KICAgICAgICBpZiBib2FyZC5oYXNfa2luZ3NpZGVfY2FzdGxpbmdfcmlnaHRzKGNoZXNzLkJMQUNLKToNCiAgICAgICAgICAgIGFjdGl2ZS5hcHBlbmQoQ0FTVExFX09XTl9LX0lEWCkNCiAgICAgICAgaWYgYm9hcmQuaGFzX3F1ZWVuc2lkZV9jYXN0bGluZ19yaWdodHMoY2hlc3MuQkxBQ0spOg0KICAgICAgICAgICAgYWN0aXZlLmFwcGVuZChDQVNUTEVfT1dOX1FfSURYKQ0KICAgICAgICBpZiBib2FyZC5oYXNfa2luZ3NpZGVfY2FzdGxpbmdfcmlnaHRzKGNoZXNzLldISVRFKToNCiAgICAgICAgICAgIGFjdGl2ZS5hcHBlbmQoQ0FTVExFX09QUF9LX0lEWCkNCiAgICAgICAgaWYgYm9hcmQuaGFzX3F1ZWVuc2lkZV9jYXN0bGluZ19yaWdodHMoY2hlc3MuV0hJVEUpOg0KICAgICAgICAgICAgYWN0aXZlLmFwcGVuZChDQVNUTEVfT1BQX1FfSURYKQ0KDQogICAgYWN0aXZlLnNvcnQoKQ0KICAgIHJldHVybiBhY3RpdmUNCg0KDQpkZWYgbGVnYWxfdG9fYnlfZnJvbShib2FyZDogY2hlc3MuQm9hcmQpIC0+IERpY3Rbc3RyLCBMaXN0W2ludF1dOg0KICAgICIiIlJldHVybiBsZWdhbCB0YXJnZXQgc3F1YXJlcyBrZXllZCBieSBub3JtYWxpemVkIGZyb20tc3F1YXJlLiIiIg0KICAgIG1hcHBpbmc6IERpY3Rbc3RyLCBMaXN0W2ludF1dID0ge30NCiAgICBtb3Zlcl9pc193aGl0ZSA9IGJvb2woYm9hcmQudHVybiA9PSBjaGVzcy5XSElURSkNCiAgICBmb3IgbXYgaW4gYm9hcmQubGVnYWxfbW92ZXM6DQogICAgICAgIHJlbF9mcm9tID0gbm9ybWFsaXplX3NxdWFyZShtdi5mcm9tX3NxdWFyZSwgbW92ZXJfaXNfd2hpdGUpDQogICAgICAgIHJlbF90byA9IG5vcm1hbGl6ZV9zcXVhcmUobXYudG9fc3F1YXJlLCBtb3Zlcl9pc193aGl0ZSkNCiAgICAgICAga2V5ID0gc3RyKHJlbF9mcm9tKQ0KICAgICAgICBpZiBrZXkgbm90IGluIG1hcHBpbmc6DQogICAgICAgICAgICBtYXBwaW5nW2tleV0gPSBbXQ0KICAgICAgICBtYXBwaW5nW2tleV0uYXBwZW5kKHJlbF90bykNCg0KICAgIGZvciBrZXkgaW4gbWFwcGluZzoNCiAgICAgICAgbWFwcGluZ1trZXldID0gc29ydGVkKHNldChtYXBwaW5nW2tleV0pKQ0KICAgIHJldHVybiBtYXBwaW5nDQoNCg0KZGVmIGxlZ2FsX2Zyb21fbWFzayhtYXBwaW5nOiBEaWN0W3N0ciwgTGlzdFtpbnRdXSkgLT4gTGlzdFtpbnRdOg0KICAgICIiIlJldHVybiA2NC1kaW0gYmluYXJ5IG1hc2sgZm9yIGxlZ2FsIGZyb20tc3F1YXJlcy4iIiINCiAgICBtYXNrID0gWzBdICogNjQNCiAgICBmb3Iga2V5IGluIG1hcHBpbmcua2V5cygpOg0KICAgICAgICBtYXNrW2ludChrZXkpXSA9IDENCiAgICByZXR1cm4gbWFzaw0KDQoNCmRlZiBoaXN0b3J5X3RvX2FycmF5cyhoaXN0b3J5OiBJdGVyYWJsZVtEaWN0W3N0ciwgQW55XV0sIGhpc3RvcnlfcGxpZXM6IGludCkgLT4gRGljdFtzdHIsIExpc3RbTGlzdFtmbG9hdF1dIHwgTGlzdFtpbnRdXToNCiAgICAiIiJQYWQvdHJ1bmNhdGUgaGlzdG9yeSBhbmQgY29udmVydCB0byBldmVudC9kZWx0YSBhcnJheXMuIiIiDQogICAgaGlzdF9saXN0ID0gbGlzdChoaXN0b3J5KVstaGlzdG9yeV9wbGllczpdDQoNCiAgICBldmVudF9yb3dzOiBMaXN0W0xpc3RbZmxvYXRdXSA9IFtdDQogICAgZGVsdGFfcm93czogTGlzdFtMaXN0W2Zsb2F0XV0gPSBbXQ0KICAgIG1hc2s6IExpc3RbaW50XSA9IFtdDQoNCiAgICBmb3IgaXRlbSBpbiBoaXN0X2xpc3Q6DQogICAgICAgIGV2ZW50ID0gaXRlbS5nZXQoImV2ZW50Iiwge30pDQogICAgICAgIGRlbHRhID0gaXRlbS5nZXQoImRlbHRhIiwge30pDQogICAgICAgIGV2ZW50X3Jvd3MuYXBwZW5kKFsNCiAgICAgICAgICAgIGZsb2F0KGV2ZW50LmdldCgibW92ZXJfaXNfdGFyZ2V0IiwgMCkpLA0KICAgICAgICAgICAgZmxvYXQoZXZlbnQuZ2V0KCJwaWVjZV90eXBlIiwgMCkpLA0KICAgICAgICAgICAgZmxvYXQoZXZlbnQuZ2V0KCJmcm9tX3NxIiwgMCkpLA0KICAgICAgICAgICAgZmxvYXQoZXZlbnQuZ2V0KCJ0b19zcSIsIDApKSwNCiAgICAgICAgICAgIGZsb2F0KGV2ZW50LmdldCgiaXNfY2FwdHVyZSIsIDApKSwNCiAgICAgICAgICAgIGZsb2F0KGV2ZW50LmdldCgiaXNfY2hlY2siLCAwKSksDQogICAgICAgICAgICBmbG9hdChldmVudC5nZXQoImlzX2Nhc3RsaW5nIiwgMCkpLA0KICAgICAgICAgICAgZmxvYXQoZXZlbnQuZ2V0KCJpc19wcm9tb3Rpb24iLCAwKSksDQogICAgICAgICAgICBmbG9hdChldmVudC5nZXQoImlzX2V4Y2hhbmdlIiwgMCkpLA0KICAgICAgICAgICAgZmxvYXQoZXZlbnQuZ2V0KCJpc19wYXduX2JyZWFrIiwgMCkpLA0KICAgICAgICAgICAgZmxvYXQoZXZlbnQuZ2V0KCJraW5nX3ByZXNzdXJlX2dhaW4iLCAwKSksDQogICAgICAgICAgICBmbG9hdChldmVudC5nZXQoInBhd25fc3RydWN0dXJlX2NoYW5nZSIsIDApKSwNCiAgICAgICAgICAgIGZsb2F0KGV2ZW50LmdldCgiaXNfc2ltcGxpZmljYXRpb24iLCAwKSksDQogICAgICAgICAgICBmbG9hdChldmVudC5nZXQoImNyZWF0ZXNfb3BwX2hhbmdpbmciLCAwKSksDQogICAgICAgICAgICBmbG9hdChldmVudC5nZXQoImNyZWF0ZXNfb3duX2hhbmdpbmciLCAwKSksDQogICAgICAgICAgICBmbG9hdChldmVudC5nZXQoImF0dGFja3NfaGlnaF92YWx1ZV9waWVjZSIsIDApKSwNCiAgICAgICAgXSkNCiAgICAgICAgZGVsdGFfcm93cy5hcHBlbmQoWw0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJtYXRlcmlhbF9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJraW5nX3NhZmV0eV9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJwYXduX3N0cnVjdHVyZV9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJtb2JpbGl0eV9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJjZW50ZXJfY29udHJvbF9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJraW5nX2FjdGl2aXR5X2RlbHRhIiwgMC4wKSksDQogICAgICAgICAgICBmbG9hdChkZWx0YS5nZXQoIm9wcF9raW5nX3ByZXNzdXJlX2RlbHRhIiwgMC4wKSksDQogICAgICAgICAgICBmbG9hdChkZWx0YS5nZXQoInRocmVhdF9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJzaW1wbGlmaWNhdGlvbl9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJvd25faGFuZ2luZ192YWx1ZV9kZWx0YSIsIDAuMCkpLA0KICAgICAgICAgICAgZmxvYXQoZGVsdGEuZ2V0KCJvcHBfaGFuZ2luZ192YWx1ZV9kZWx0YSIsIDAuMCkpLA0KICAgICAgICBdKQ0KICAgICAgICBtYXNrLmFwcGVuZCgxKQ0KDQogICAgcGFkID0gaGlzdG9yeV9wbGllcyAtIGxlbihldmVudF9yb3dzKQ0KICAgIGlmIHBhZCA+IDA6DQogICAgICAgIGV2ZW50X3Jvd3MgPSBbWzAuMF0gKiBISVNUT1JZX0VWRU5UX0RJTSBmb3IgXyBpbiByYW5nZShwYWQpXSArIGV2ZW50X3Jvd3MNCiAgICAgICAgZGVsdGFfcm93cyA9IFtbMC4wXSAqIEhJU1RPUllfREVMVEFfRElNIGZvciBfIGluIHJhbmdlKHBhZCldICsgZGVsdGFfcm93cw0KICAgICAgICBtYXNrID0gWzBdICogcGFkICsgbWFzaw0KDQogICAgcmV0dXJuIHsNCiAgICAgICAgImhpc3RvcnlfZXZlbnQiOiBldmVudF9yb3dzLA0KICAgICAgICAiaGlzdG9yeV9kZWx0YSI6IGRlbHRhX3Jvd3MsDQogICAgICAgICJoaXN0b3J5X21hc2siOiBtYXNrLA0KICAgIH0NCg0KDQpkZWYgYnVpbGRfZGVuc2Vfc3RhdGUoYm9hcmQ6IGNoZXNzLkJvYXJkLCByb3c6IERpY3Rbc3RyLCBBbnldKSAtPiBMaXN0W2Zsb2F0XToNCiAgICAiIiJCdWlsZCBjb21wYWN0IGRlbnNlIHN0YXRlIHN1bW1hcnkgZmVhdHVyZXMgZm9yIHRoZSBjdXJyZW50IHBvc2l0aW9uLiIiIg0KICAgIHJldHVybiBkZW5zZV9zdGF0ZV92ZWN0b3IoDQogICAgICAgIGJvYXJkPWJvYXJkLA0KICAgICAgICB0YXJnZXRfY29sb3I9Ym9vbChib2FyZC50dXJuKSwNCiAgICAgICAgcGhhc2VfY29kZT1pbnQocm93LmdldCgicGhhc2VfY29kZSIsIDApKSwNCiAgICAgICAgZnVsbG1vdmVfbnVtYmVyPWludChyb3cuZ2V0KCJmdWxsbW92ZV9udW1iZXIiLCBib2FyZC5mdWxsbW92ZV9udW1iZXIpKSwNCiAgICApDQoNCg0KZGVmIGl0ZXJfanNvbmwocGF0aDogUGF0aCkgLT4gSXRlcmFibGVbRGljdFtzdHIsIEFueV1dOg0KICAgICIiIllpZWxkIEpTT05MIHJvd3MgYXMgZGljdGlvbmFyaWVzLiIiIg0KICAgIHdpdGggcGF0aC5vcGVuKCJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOg0KICAgICAgICBmb3IgbGluZSBpbiBoYW5kbGU6DQogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpDQogICAgICAgICAgICBpZiBsaW5lOg0KICAgICAgICAgICAgICAgIHlpZWxkIGpzb24ubG9hZHMobGluZSkNCg0KDQpkZWYgZW5jb2RlX3Jvdyhyb3c6IERpY3Rbc3RyLCBBbnldLCBoaXN0b3J5X3BsaWVzOiBpbnQpIC0+IERpY3Rbc3RyLCBBbnldIHwgTm9uZToNCiAgICAiIiJFbmNvZGUgb25lIHBhcnNlZCBwb3NpdGlvbiByb3cgaW50byBhIHBvbGljeSB0cmFpbmluZyBzYW1wbGUuIiIiDQogICAgZmVuID0gc3RyKHJvd1siZmVuX2JlZm9yZSJdKQ0KICAgIGJvYXJkID0gY2hlc3MuQm9hcmQoZmVuKQ0KDQogICAgbGVnYWxfbWFwID0gbGVnYWxfdG9fYnlfZnJvbShib2FyZCkNCiAgICBoaXN0b3J5X2FycmF5cyA9IGhpc3RvcnlfdG9fYXJyYXlzKHJvdy5nZXQoImhpc3RvcnkiLCBbXSksIGhpc3RvcnlfcGxpZXMpDQogICAgbW92ZWRfcGllY2Vfc2xvdCA9IHN0cihyb3cuZ2V0KCJtb3ZlZF9waWVjZV9zbG90Iikgb3IgIiIpLnN0cmlwKCkNCiAgICBpZiBtb3ZlZF9waWVjZV9zbG90IG5vdCBpbiBPUklHSU5BTF9TTE9UX1RPX0lOREVYOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgIHBpZWNlX3Nsb3Rfc3F1YXJlcyA9IHBpZWNlX3Nsb3Rfc3F1YXJlX2xpc3Qocm93LmdldCgicGllY2Vfc2xvdF90b19zcXVhcmUiLCB7fSkpDQoNCiAgICBzYW1wbGUgPSB7DQogICAgICAgICJnYW1lX2lkIjogcm93LmdldCgiZ2FtZV9pZCIpLA0KICAgICAgICAicGxheWVyX2lkIjogc3RyKHJvdy5nZXQoInBsYXllcl9pZCIpIG9yIHJvdy5nZXQoInRhcmdldF91c2VybmFtZSIpIG9yICJ1bmtub3duIikuc3RyaXAoKS5sb3dlcigpLA0KICAgICAgICAicGx5X2luZGV4IjogaW50KHJvdy5nZXQoInBseV9pbmRleCIsIDApKSwNCiAgICAgICAgImZlbl9iZWZvcmUiOiBmZW4sDQogICAgICAgICJhY3RpdmVfZmVhdHVyZV9pbmRpY2VzIjogZW5jb2RlX2JvYXJkX3NwYXJzZV9pbmRpY2VzKGZlbiksDQogICAgICAgICJkZW5zZV9zdGF0ZSI6IGJ1aWxkX2RlbnNlX3N0YXRlKGJvYXJkLCByb3cpLA0KICAgICAgICAiaGlzdG9yeV9ldmVudCI6IGhpc3RvcnlfYXJyYXlzWyJoaXN0b3J5X2V2ZW50Il0sDQogICAgICAgICJoaXN0b3J5X2RlbHRhIjogaGlzdG9yeV9hcnJheXNbImhpc3RvcnlfZGVsdGEiXSwNCiAgICAgICAgImhpc3RvcnlfbWFzayI6IGhpc3RvcnlfYXJyYXlzWyJoaXN0b3J5X21hc2siXSwNCiAgICAgICAgImNvbnRleHQiOiBbDQogICAgICAgICAgICBmbG9hdChyb3cuZ2V0KCJ0YXJnZXRfaXNfd2hpdGUiLCAwKSksDQogICAgICAgIF0sDQogICAgICAgICJwaWVjZV9zbG90X3RvX3NxdWFyZSI6IHBpZWNlX3Nsb3Rfc3F1YXJlcywNCiAgICAgICAgImxlZ2FsX3BpZWNlX3Nsb3RfbWFzayI6IGxlZ2FsX3BpZWNlX3Nsb3RfbWFzayhwaWVjZV9zbG90X3NxdWFyZXMsIGxlZ2FsX21hcCksDQogICAgICAgICJ0YXJnZXRfcGllY2Vfc2xvdCI6IGludChPUklHSU5BTF9TTE9UX1RPX0lOREVYW21vdmVkX3BpZWNlX3Nsb3RdKSwNCiAgICAgICAgImxlZ2FsX2Zyb21fbWFzayI6IGxlZ2FsX2Zyb21fbWFzayhsZWdhbF9tYXApLA0KICAgICAgICAibGVnYWxfdG9fYnlfZnJvbSI6IGxlZ2FsX21hcCwNCiAgICAgICAgInRhcmdldF9mcm9tX3NxIjogaW50KHJvd1sidGFyZ2V0X2Zyb21fc3EiXSksDQogICAgICAgICJ0YXJnZXRfdG9fc3EiOiBpbnQocm93WyJ0YXJnZXRfdG9fc3EiXSksDQogICAgICAgICJ0YXJnZXRfcHJvbW90aW9uIjogaW50KHJvdy5nZXQoInRhcmdldF9wcm9tb3Rpb24iLCAwKSksDQogICAgICAgICJ0YXJnZXRfdW5kZXJfdGhyZWF0IjogaW50KGlzX3VuZGVyX2ltbWVkaWF0ZV90aHJlYXQoYm9hcmQsIGJvb2woYm9hcmQudHVybikpKSwNCiAgICAgICAgInRhcmdldF91Y2kiOiByb3cuZ2V0KCJwbGF5ZWRfdWNpIiksDQogICAgfQ0KICAgIHJldHVybiBzYW1wbGUNCg0KDQpkZWYgbWFpbigpIC0+IE5vbmU6DQogICAgIiIiUnVuIHN0YWdlIDIgYW5kIHdyaXRlIGVuY29kZWQgcG9saWN5IHNhbXBsZXMgYXMgSlNPTkwuIiIiDQogICAgaW5fcGF0aCA9IFBhdGgoQ09ORklHWyJpbnB1dF9qc29ubCJdKQ0KICAgIG91dF9wYXRoID0gUGF0aChDT05GSUdbIm91dHB1dF9qc29ubCJdKQ0KICAgIGhpc3RvcnlfcGxpZXMgPSBpbnQoQ09ORklHWyJoaXN0b3J5X3BsaWVzIl0pDQogICAgcHJvZ3Jlc3NfZXZlcnkgPSBpbnQoQ09ORklHLmdldCgicHJvZ3Jlc3NfZXZlcnkiLCAwKSkNCg0KICAgIGlmIG5vdCBpbl9wYXRoLmV4aXN0cygpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIklucHV0IGZpbGUgbm90IGZvdW5kOiB7aW5fcGF0aH0iKQ0KDQogICAgb3V0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCg0KICAgIGNvdW50ID0gMA0KICAgIHdpdGggb3V0X3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIG91dDoNCiAgICAgICAgZm9yIHJvdyBpbiBpdGVyX2pzb25sKGluX3BhdGgpOg0KICAgICAgICAgICAgc2FtcGxlID0gZW5jb2RlX3Jvdyhyb3csIGhpc3RvcnlfcGxpZXMpDQogICAgICAgICAgICBpZiBzYW1wbGUgaXMgTm9uZToNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgb3V0LndyaXRlKGpzb24uZHVtcHMoc2FtcGxlLCBlbnN1cmVfYXNjaWk9RmFsc2UpICsgIlxuIikNCiAgICAgICAgICAgIGNvdW50ICs9IDENCiAgICAgICAgICAgIGlmIGJvb2woQ09ORklHLmdldCgidmVyYm9zZSIsIFRydWUpKSBhbmQgcHJvZ3Jlc3NfZXZlcnkgPiAwIGFuZCBjb3VudCAlIHByb2dyZXNzX2V2ZXJ5ID09IDA6DQogICAgICAgICAgICAgICAgcHJpbnQoZiJFbmNvZGVkIHtjb3VudH0gc2FtcGxlcy4uLiIpDQoNCiAgICBpZiBib29sKENPTkZJRy5nZXQoInZlcmJvc2UiLCBUcnVlKSk6DQogICAgICAgIHByaW50KGYiRW5jb2RlZCB7Y291bnR9IHNhbXBsZXMgLT4ge291dF9wYXRofSIpDQogICAgICAgIHByaW50KGYiRmVhdHVyZSB2b2NhYiBzaXplIChIYWxmS1ApOiB7VE9UQUxfRkVBVFVSRV9ESU19IikNCiAgICAgICAgcHJpbnQoZiJEZW5zZSBzdGF0ZSBkaW06IHtERU5TRV9TVEFURV9ESU19IHwgaGlzdG9yeSBldmVudCBkaW06IHtISVNUT1JZX0VWRU5UX0RJTX0gfCBoaXN0b3J5IGRlbHRhIGRpbToge0hJU1RPUllfREVMVEFfRElNfSIpDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICBtYWluKCkNCg0KDQoNCg==",
  "scripts/script3_pretrain_history_policy.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiUHJldHJhaW4gcmVwcmVzZW50YXRpb24gZW5jb2RlcnMgb24gbXVsdGktcGxheWVyIHBvbGljeSBzYW1wbGVzLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBqc29uDQppbXBvcnQgcmFuZG9tDQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIERpY3QsIExpc3QNCg0KaW1wb3J0IHRvcmNoDQoNCmZyb20gaGlzdG9yeV9wb2xpY3lfbGliIGltcG9ydCAoDQogICAgRmFjdG9yaXplZFBvbGljeU1vZGVsLA0KICAgIGV2YWx1YXRlLA0KICAgIGxvYWRfc2FtcGxlcywNCiAgICBzYXZlX2NoZWNrcG9pbnQsDQogICAgc2V0X3NlZWQsDQogICAgc3BsaXRfYnlfZ2FtZSwNCiAgICB0cmFpbl9vbmVfZXBvY2gsDQopDQpmcm9tIHBpcGVsaW5lX2NvbmZpZyBpbXBvcnQgSElTVE9SWV9QTElFUywgUFJFVFJBSU5fREFUQVNFVF9UQUcsIFRBUkdFVF9VU0VSTkFNRSwgbW9kZWxzX2RpciwgcHJvY2Vzc2VkX2Rpcg0KDQoNCmRlZiBub3JtYWxpemVfcGxheWVyX2lkKG5hbWU6IHN0cikgLT4gc3RyOg0KICAgICIiIk5vcm1hbGl6ZSBwbGF5ZXIgbmFtZSBpbnRvIHBsYXllcl9pZCBmb3JtYXQgdXNlZCBpbiBlbmNvZGVkIHNhbXBsZXMuIiIiDQogICAgcmV0dXJuICIgIi5qb2luKG5hbWUuc3RyaXAoKS5zcGxpdCgpKS5sb3dlcigpDQoNCg0KQ09ORklHOiBEaWN0W3N0ciwgQW55XSA9IHsNCiAgICAiZGF0YXNldF90YWciOiBQUkVUUkFJTl9EQVRBU0VUX1RBRywNCiAgICAiaGlzdG9yeV9wbGllcyI6IEhJU1RPUllfUExJRVMsDQogICAgImlucHV0X2pzb25sIjogcHJvY2Vzc2VkX2RpcihQUkVUUkFJTl9EQVRBU0VUX1RBRykgLyAicG9saWN5X3NhbXBsZXMuanNvbmwiLA0KICAgICJtb2RlbF9vdXRwdXRfcGF0aCI6IG1vZGVsc19kaXIoUFJFVFJBSU5fREFUQVNFVF9UQUcpIC8gImhpc3RvcnlfcG9saWN5X3ByZXRyYWluLnB0IiwNCiAgICAibWV0cmljc19vdXRwdXRfcGF0aCI6IG1vZGVsc19kaXIoUFJFVFJBSU5fREFUQVNFVF9UQUcpIC8gImhpc3RvcnlfcG9saWN5X3ByZXRyYWluX21ldHJpY3MuanNvbiIsDQogICAgInZhbGlkX3NpemUiOiAwLjEwLA0KICAgICJyYW5kb21fc2VlZCI6IDQyLA0KICAgICJlcG9jaHMiOiA4LA0KICAgICJlcG9jaHNfY3B1IjogNCwNCiAgICAiYmF0Y2hfc2l6ZSI6IDI1NiwNCiAgICAiYmF0Y2hfc2l6ZV9jcHUiOiA0OCwNCiAgICAicHJpbnRfZXZlcnkiOiAyMCwNCiAgICAiZXZhbF9iYXRjaF9zaXplIjogMjU2LA0KICAgICJldmFsX2JhdGNoX3NpemVfY3B1IjogMTI4LA0KICAgICJncmFkX2FjY3VtX3N0ZXBzIjogMSwNCiAgICAiZ3JhZF9hY2N1bV9zdGVwc19jcHUiOiAyLA0KICAgICJsZWFybmluZ19yYXRlIjogMWUtMywNCiAgICAid2VpZ2h0X2RlY2F5IjogMWUtNSwNCiAgICAiZmVhdHVyZV92b2NhYl9zaXplIjogODE5MjQsDQogICAgImZlYXR1cmVfZW1iZWRfZGltIjogNjQsDQogICAgImRlbnNlX3N0YXRlX2RpbSI6IDI5LA0KICAgICJjb250ZXh0X2RpbSI6IDEsDQogICAgImhpc3RvcnlfZXZlbnRfZGltIjogMTYsDQogICAgImhpc3RvcnlfZGVsdGFfZGltIjogMTEsDQogICAgImhpc3RvcnlfaGlkZGVuX2RpbSI6IDk2LA0KICAgICJzaGFyZWRfaGlkZGVuX2RpbSI6IDEyOCwNCiAgICAiZHJvcG91dCI6IDAuMTAsDQogICAgImVuYWJsZV90aHJlYXRfaGVhZCI6IFRydWUsDQogICAgInRocmVhdF9sb3NzX3dlaWdodCI6IDAuMiwNCiAgICAiZGV2aWNlIjogImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IiwNCiAgICAjIFNldCBzdHJpY3RfdGFyZ2V0X2lzb2xhdGlvbj1UcnVlIHRvIHJlbW92ZSB0YXJnZXQgcGxheWVyIGZyb20gcHJldHJhaW5pbmcgZGF0YS4NCiAgICAic3RyaWN0X3RhcmdldF9pc29sYXRpb24iOiBGYWxzZSwNCiAgICAibWF4X3NhbXBsZXMiOiBOb25lLA0KICAgICJtYXhfc2FtcGxlc19jcHUiOiAyNTAwMDAsDQogICAgInJlc2Vydm9pcl9zYW1wbGVfZm9yX2NhcCI6IFRydWUsDQogICAgImxvYWRfcHJvZ3Jlc3NfZXZlcnkiOiA1MDAwMCwNCiAgICAidHJhaW5fZXZhbF9tYXhfc2FtcGxlcyI6IDUwMDAwLA0KICAgICJ2YWxpZF9ldmFsX21heF9zYW1wbGVzIjogNTAwMDAsDQogICAgImV4Y2x1ZGVkX3BsYXllcl9pZHMiOiBbbm9ybWFsaXplX3BsYXllcl9pZChUQVJHRVRfVVNFUk5BTUUpXSwNCn0NCg0KDQpkZWYgbWF5YmVfZmlsdGVyX3NhbXBsZXMoc2FtcGxlczogTGlzdFtBbnldLCBjb25maWc6IERpY3Rbc3RyLCBBbnldKSAtPiBMaXN0W0FueV06DQogICAgIiIiT3B0aW9uYWxseSByZW1vdmUgZXhjbHVkZWQgcGxheWVycyBmcm9tIHByZXRyYWluaW5nIHNhbXBsZXMuIiIiDQogICAgaWYgbm90IGJvb2woY29uZmlnLmdldCgic3RyaWN0X3RhcmdldF9pc29sYXRpb24iLCBGYWxzZSkpOg0KICAgICAgICByZXR1cm4gc2FtcGxlcw0KDQogICAgZXhjbHVkZWQgPSB7c3RyKHgpLnN0cmlwKCkubG93ZXIoKSBmb3IgeCBpbiBjb25maWcuZ2V0KCJleGNsdWRlZF9wbGF5ZXJfaWRzIiwgW10pIGlmIHN0cih4KS5zdHJpcCgpfQ0KICAgIGlmIG5vdCBleGNsdWRlZDoNCiAgICAgICAgcmV0dXJuIHNhbXBsZXMNCg0KICAgIGtlcHQgPSBbcyBmb3IgcyBpbiBzYW1wbGVzIGlmIHN0cihnZXRhdHRyKHMsICJwbGF5ZXJfaWQiLCAiIikpLnN0cmlwKCkubG93ZXIoKSBub3QgaW4gZXhjbHVkZWRdDQogICAgcmVtb3ZlZCA9IGxlbihzYW1wbGVzKSAtIGxlbihrZXB0KQ0KICAgIHByaW50KGYiU3RyaWN0IGlzb2xhdGlvbiBlbmFibGVkOiByZW1vdmVkIHtyZW1vdmVkfSBzYW1wbGVzIGZyb20gZXhjbHVkZWQgcGxheWVyczoge3NvcnRlZChleGNsdWRlZCl9IikNCiAgICBpZiBub3Qga2VwdDoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQWxsIHNhbXBsZXMgcmVtb3ZlZCBieSBzdHJpY3RfdGFyZ2V0X2lzb2xhdGlvbiBmaWx0ZXIiKQ0KICAgIHJldHVybiBrZXB0DQoNCg0KZGVmIG1heWJlX3N1YnNhbXBsZShzYW1wbGVzOiBMaXN0W0FueV0sIG1heF9uOiBpbnQgfCBOb25lLCBzZWVkOiBpbnQpIC0+IExpc3RbQW55XToNCiAgICAiIiJSYW5kb21seSBzdWJzYW1wbGUgbGlzdCBmb3IgZmFzdGVyL2xvd2VyLW1lbW9yeSBldmFsdWF0aW9uLiIiIg0KICAgIGlmIG1heF9uIGlzIE5vbmUgb3IgbWF4X24gPD0gMCBvciBsZW4oc2FtcGxlcykgPD0gbWF4X246DQogICAgICAgIHJldHVybiBzYW1wbGVzDQogICAgcm5nID0gcmFuZG9tLlJhbmRvbShzZWVkKQ0KICAgIGlkeCA9IHJuZy5zYW1wbGUocmFuZ2UobGVuKHNhbXBsZXMpKSwgaW50KG1heF9uKSkNCiAgICByZXR1cm4gW3NhbXBsZXNbaV0gZm9yIGkgaW4gaWR4XQ0KDQoNCmRlZiBtYWluKCkgLT4gTm9uZToNCiAgICAiIiJSdW4gbXVsdGktcGxheWVyIHByZXRyYWluaW5nIGFuZCBzYXZlIGJlc3QgY2hlY2twb2ludCBieSB2YWxpZCBtb3ZlIGFjY3VyYWN5LiIiIg0KICAgIHNldF9zZWVkKGludChDT05GSUdbInJhbmRvbV9zZWVkIl0pKQ0KDQogICAgaW5fcGF0aCA9IFBhdGgoQ09ORklHWyJpbnB1dF9qc29ubCJdKQ0KICAgIGlmIG5vdCBpbl9wYXRoLmV4aXN0cygpOg0KICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIklucHV0IGZpbGUgbm90IGZvdW5kOiB7aW5fcGF0aH0iKQ0KDQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKHN0cihDT05GSUdbImRldmljZSJdKSkNCiAgICBpc19jcHUgPSBkZXZpY2UudHlwZSA9PSAiY3B1Ig0KDQogICAgbWF4X3NhbXBsZXNfcmF3ID0gQ09ORklHLmdldCgibWF4X3NhbXBsZXNfY3B1IikgaWYgaXNfY3B1IGVsc2UgQ09ORklHLmdldCgibWF4X3NhbXBsZXMiKQ0KICAgIG1heF9zYW1wbGVzID0gaW50KG1heF9zYW1wbGVzX3JhdykgaWYgbWF4X3NhbXBsZXNfcmF3IGlzIG5vdCBOb25lIGVsc2UgTm9uZQ0KDQogICAgc2FtcGxlcyA9IGxvYWRfc2FtcGxlcygNCiAgICAgICAgaW5fcGF0aCwNCiAgICAgICAgbWF4X3NhbXBsZXM9bWF4X3NhbXBsZXMsDQogICAgICAgIHJlc2Vydm9pcj1ib29sKENPTkZJRy5nZXQoInJlc2Vydm9pcl9zYW1wbGVfZm9yX2NhcCIsIEZhbHNlKSksDQogICAgICAgIHNlZWQ9aW50KENPTkZJR1sicmFuZG9tX3NlZWQiXSksDQogICAgICAgIHByb2dyZXNzX2V2ZXJ5PWludChDT05GSUcuZ2V0KCJsb2FkX3Byb2dyZXNzX2V2ZXJ5IiwgMCkpLA0KICAgICAgICBwcm9ncmVzc19sYWJlbD1mIntDT05GSUdbJ2RhdGFzZXRfdGFnJ119OmxvYWQiLA0KICAgICkNCiAgICBpZiBtYXhfc2FtcGxlcyBpcyBub3QgTm9uZToNCiAgICAgICAgcHJpbnQoZiJMb2FkZWQgY2FwcGVkIHNhbXBsZSBzZXQgZm9yIHRyYWluaW5nOiB7bGVuKHNhbXBsZXMpfSByb3dzIChtYXhfc2FtcGxlcz17bWF4X3NhbXBsZXN9KSIpDQogICAgZWxzZToNCiAgICAgICAgcHJpbnQoZiJMb2FkZWQgZnVsbCBzYW1wbGUgc2V0IGZvciB0cmFpbmluZzoge2xlbihzYW1wbGVzKX0gcm93cyIpDQoNCiAgICBzYW1wbGVzID0gbWF5YmVfZmlsdGVyX3NhbXBsZXMoc2FtcGxlcywgQ09ORklHKQ0KICAgIHRyYWluX3NhbXBsZXMsIHZhbGlkX3NhbXBsZXMgPSBzcGxpdF9ieV9nYW1lKHNhbXBsZXMsIGZsb2F0KENPTkZJR1sidmFsaWRfc2l6ZSJdKSwgaW50KENPTkZJR1sicmFuZG9tX3NlZWQiXSkpDQoNCiAgICBlZmZlY3RpdmVfZXBvY2hzID0gaW50KENPTkZJR1siZXBvY2hzX2NwdSJdIGlmIGlzX2NwdSBlbHNlIENPTkZJR1siZXBvY2hzIl0pDQogICAgYmF0Y2hfc2l6ZSA9IGludChDT05GSUdbImJhdGNoX3NpemVfY3B1Il0gaWYgaXNfY3B1IGVsc2UgQ09ORklHWyJiYXRjaF9zaXplIl0pDQogICAgZXZhbF9iYXRjaF9zaXplID0gaW50KENPTkZJR1siZXZhbF9iYXRjaF9zaXplX2NwdSJdIGlmIGlzX2NwdSBlbHNlIENPTkZJR1siZXZhbF9iYXRjaF9zaXplIl0pDQogICAgZ3JhZF9hY2N1bV9zdGVwcyA9IGludChDT05GSUdbImdyYWRfYWNjdW1fc3RlcHNfY3B1Il0gaWYgaXNfY3B1IGVsc2UgQ09ORklHWyJncmFkX2FjY3VtX3N0ZXBzIl0pDQoNCiAgICBtb2RlbCA9IEZhY3Rvcml6ZWRQb2xpY3lNb2RlbChDT05GSUcpLnRvKGRldmljZSkNCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVygNCiAgICAgICAgbW9kZWwucGFyYW1ldGVycygpLA0KICAgICAgICBscj1mbG9hdChDT05GSUdbImxlYXJuaW5nX3JhdGUiXSksDQogICAgICAgIHdlaWdodF9kZWNheT1mbG9hdChDT05GSUdbIndlaWdodF9kZWNheSJdKSwNCiAgICApDQoNCiAgICBiZXN0X21vdmVfYWNjID0gLTEuMA0KICAgIGJlc3RfbWV0cmljczogRGljdFtzdHIsIEFueV0gPSB7fQ0KDQogICAgZm9yIGVwb2NoIGluIHJhbmdlKDEsIGVmZmVjdGl2ZV9lcG9jaHMgKyAxKToNCiAgICAgICAgdHJhaW5fbG9zcyA9IHRyYWluX29uZV9lcG9jaCgNCiAgICAgICAgICAgIG1vZGVsPW1vZGVsLA0KICAgICAgICAgICAgc2FtcGxlcz10cmFpbl9zYW1wbGVzLA0KICAgICAgICAgICAgb3B0aW1pemVyPW9wdGltaXplciwNCiAgICAgICAgICAgIGRldmljZT1kZXZpY2UsDQogICAgICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsDQogICAgICAgICAgICBwcmludF9ldmVyeT1pbnQoQ09ORklHWyJwcmludF9ldmVyeSJdKSwNCiAgICAgICAgICAgIGVwb2NoX2luZGV4PWVwb2NoLA0KICAgICAgICAgICAgdG90YWxfZXBvY2hzPWVmZmVjdGl2ZV9lcG9jaHMsDQogICAgICAgICAgICBncmFkX2FjY3VtX3N0ZXBzPWdyYWRfYWNjdW1fc3RlcHMsDQogICAgICAgICkNCg0KICAgICAgICBldmFsX3RyYWluX3NhbXBsZXMgPSBtYXliZV9zdWJzYW1wbGUoDQogICAgICAgICAgICB0cmFpbl9zYW1wbGVzLA0KICAgICAgICAgICAgaW50KENPTkZJR1sidHJhaW5fZXZhbF9tYXhfc2FtcGxlcyJdKSBpZiBDT05GSUcuZ2V0KCJ0cmFpbl9ldmFsX21heF9zYW1wbGVzIikgaXMgbm90IE5vbmUgZWxzZSBOb25lLA0KICAgICAgICAgICAgc2VlZD1pbnQoQ09ORklHWyJyYW5kb21fc2VlZCJdKSArIGVwb2NoLA0KICAgICAgICApDQogICAgICAgIGV2YWxfdmFsaWRfc2FtcGxlcyA9IG1heWJlX3N1YnNhbXBsZSgNCiAgICAgICAgICAgIHZhbGlkX3NhbXBsZXMsDQogICAgICAgICAgICBpbnQoQ09ORklHWyJ2YWxpZF9ldmFsX21heF9zYW1wbGVzIl0pIGlmIENPTkZJRy5nZXQoInZhbGlkX2V2YWxfbWF4X3NhbXBsZXMiKSBpcyBub3QgTm9uZSBlbHNlIE5vbmUsDQogICAgICAgICAgICBzZWVkPWludChDT05GSUdbInJhbmRvbV9zZWVkIl0pICsgMTAwMCArIGVwb2NoLA0KICAgICAgICApDQoNCiAgICAgICAgdHJhaW5fbWV0cmljcyA9IGV2YWx1YXRlKG1vZGVsLCBldmFsX3RyYWluX3NhbXBsZXMsIGRldmljZT1kZXZpY2UsIGJhdGNoX3NpemU9ZXZhbF9iYXRjaF9zaXplKQ0KICAgICAgICB2YWxpZF9tZXRyaWNzID0gZXZhbHVhdGUobW9kZWwsIGV2YWxfdmFsaWRfc2FtcGxlcywgZGV2aWNlPWRldmljZSwgYmF0Y2hfc2l6ZT1ldmFsX2JhdGNoX3NpemUpDQoNCiAgICAgICAgcHJpbnQoDQogICAgICAgICAgICBmIkVwb2NoIHtlcG9jaDowMmR9IGRvbmUgfCAiDQogICAgICAgICAgICBmImxvc3M9e3RyYWluX2xvc3M6LjRmfSB8ICINCiAgICAgICAgICAgIGYidHJhaW5fbW92ZV9hY2M9e3RyYWluX21ldHJpY3NbJ21vdmVfYWNjJ106LjRmfSB8ICINCiAgICAgICAgICAgIGYidmFsaWRfbW92ZV9hY2M9e3ZhbGlkX21ldHJpY3NbJ21vdmVfYWNjJ106LjRmfSINCiAgICAgICAgKQ0KDQogICAgICAgIGlmIHZhbGlkX21ldHJpY3NbIm1vdmVfYWNjIl0gPiBiZXN0X21vdmVfYWNjOg0KICAgICAgICAgICAgYmVzdF9tb3ZlX2FjYyA9IHZhbGlkX21ldHJpY3NbIm1vdmVfYWNjIl0NCiAgICAgICAgICAgIGJlc3RfbWV0cmljcyA9IHsNCiAgICAgICAgICAgICAgICAiYmVzdF9lcG9jaCI6IGVwb2NoLA0KICAgICAgICAgICAgICAgICJ0cmFpbiI6IHRyYWluX21ldHJpY3MsDQogICAgICAgICAgICAgICAgInZhbGlkIjogdmFsaWRfbWV0cmljcywNCiAgICAgICAgICAgICAgICAiaGlzdG9yeV9wbGllcyI6IGludChDT05GSUdbImhpc3RvcnlfcGxpZXMiXSksDQogICAgICAgICAgICAgICAgIm1vZGUiOiAicHJldHJhaW4iLA0KICAgICAgICAgICAgICAgICJzdHJpY3RfdGFyZ2V0X2lzb2xhdGlvbiI6IGJvb2woQ09ORklHLmdldCgic3RyaWN0X3RhcmdldF9pc29sYXRpb24iLCBGYWxzZSkpLA0KICAgICAgICAgICAgfQ0KICAgICAgICAgICAgc2F2ZV9jaGVja3BvaW50KG1vZGVsLCBkaWN0KENPTkZJRyksIGJlc3RfbWV0cmljcywgUGF0aChDT05GSUdbIm1vZGVsX291dHB1dF9wYXRoIl0pKQ0KDQogICAgbWV0cmljc19wYXRoID0gUGF0aChDT05GSUdbIm1ldHJpY3Nfb3V0cHV0X3BhdGgiXSkNCiAgICBtZXRyaWNzX3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB3aXRoIG1ldHJpY3NfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOg0KICAgICAgICBqc29uLmR1bXAoYmVzdF9tZXRyaWNzLCBoYW5kbGUsIGluZGVudD0yKQ0KDQogICAgcHJpbnQoZiJCZXN0IHZhbGlkIG1vdmVfYWNjOiB7YmVzdF9tb3ZlX2FjYzouNGZ9IikNCiAgICBwcmludChmIk1vZGVsIHNhdmVkOiB7Q09ORklHWydtb2RlbF9vdXRwdXRfcGF0aCddfSIpDQogICAgcHJpbnQoZiJNZXRyaWNzIHNhdmVkOiB7Q09ORklHWydtZXRyaWNzX291dHB1dF9wYXRoJ119IikNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0KDQoNCg0KDQo=",
  "scripts/script3_pretrain_history_policy_stream.py": "...",
  "scripts/script4_finetune_history_policy.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMw0KIiIiRmluZS10dW5lIHRhcmdldC1wbGF5ZXIgcG9saWN5IGZyb20gb3B0aW9uYWwgcmVwcmVzZW50YXRpb24gcHJldHJhaW5pbmcuIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KaW1wb3J0IGpzb24NCmltcG9ydCByYW5kb20NCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgRGljdCwgTGlzdCwgVHVwbGUNCg0KaW1wb3J0IHRvcmNoDQoNCmZyb20gaGlzdG9yeV9wb2xpY3lfbGliIGltcG9ydCAoDQogICAgRmFjdG9yaXplZFBvbGljeU1vZGVsLA0KICAgIGV2YWx1YXRlLA0KICAgIGxvYWRfY2hlY2twb2ludCwNCiAgICBsb2FkX3ByZXRyYWluZWRfZW5jb2RlcnMsDQogICAgbG9hZF9zYW1wbGVzLA0KICAgIHNhdmVfY2hlY2twb2ludCwNCiAgICBzZXRfc2VlZCwNCiAgICB0cmFpbl9vbmVfZXBvY2gsDQopDQpmcm9tIHBpcGVsaW5lX2NvbmZpZyBpbXBvcnQgREFUQVNFVF9UQUcsIEhJU1RPUllfUExJRVMsIG1vZGVsc19kaXIsIHByb2Nlc3NlZF9kaXINCg0KDQpDT05GSUc6IERpY3Rbc3RyLCBBbnldID0gew0KICAgICJkYXRhc2V0X3RhZyI6IERBVEFTRVRfVEFHLA0KICAgICJoaXN0b3J5X3BsaWVzIjogSElTVE9SWV9QTElFUywNCiAgICAiaW5wdXRfanNvbmwiOiBwcm9jZXNzZWRfZGlyKERBVEFTRVRfVEFHKSAvICJwb2xpY3lfc2FtcGxlcy5qc29ubCIsDQogICAgIm1vZGVsX291dHB1dF9wYXRoIjogbW9kZWxzX2RpcihEQVRBU0VUX1RBRykgLyAiaGlzdG9yeV9wb2xpY3kucHQiLCAjIGhpc3RvcnlfcG9saWN5X3NjcmF0Y2gNCiAgICAibWV0cmljc19vdXRwdXRfcGF0aCI6IG1vZGVsc19kaXIoREFUQVNFVF9UQUcpIC8gImhpc3RvcnlfcG9saWN5X21ldHJpY3MuanNvbiIsICMgaGlzdG9yeV9wb2xpY3lfc2NyYXRtZXRyaWNzDQogICAgImhvbmVzdF9zcGxpdF9vdXRwdXRfcGF0aCI6IG1vZGVsc19kaXIoREFUQVNFVF9UQUcpIC8gImhvbmVzdF9zcGxpdF9nYW1lX2lkcy5qc29uIiwgIyBob25lc3Rfc3BsaXRfZ2FtZV9pZHNfc2NyYXQNCiAgICAicHJldHJhaW5lZF9wYXRoIjogbW9kZWxzX2RpcigicHJldHJhaW5fbXVsdGkiKSAvICJoaXN0b3J5X3BvbGljeV9wcmV0cmFpbl9zdHJlYW0ucHQiLA0KICAgICJ1c2VfcHJldHJhaW5lZF9lbmNvZGVycyI6IFRydWUsICMgRmFsc2Ugbm8gcHJldHJhaW4NCiAgICAidmFsaWRfc2l6ZSI6IDAuMTAsDQogICAgInRlc3Rfc2l6ZSI6IDAuMTAsDQogICAgInJhbmRvbV9zZWVkIjogNDIsDQogICAgImVwb2NocyI6IDEyLA0KICAgICJlcG9jaHNfY3B1IjogOCwNCiAgICAiYmF0Y2hfc2l6ZSI6IDE5MiwNCiAgICAiYmF0Y2hfc2l6ZV9jcHUiOiA2NCwNCiAgICAicHJpbnRfZXZlcnkiOiAyMCwNCiAgICAiZXZhbF9iYXRjaF9zaXplIjogMTkyLA0KICAgICJldmFsX2JhdGNoX3NpemVfY3B1IjogMTI4LA0KICAgICJncmFkX2FjY3VtX3N0ZXBzIjogMSwNCiAgICAiZ3JhZF9hY2N1bV9zdGVwc19jcHUiOiAyLA0KICAgICJtYXhfc2FtcGxlcyI6IE5vbmUsDQogICAgIm1heF9zYW1wbGVzX2NwdSI6IDIwMDAwMCwNCiAgICAicmVzZXJ2b2lyX3NhbXBsZV9mb3JfY2FwIjogVHJ1ZSwNCiAgICAibG9hZF9wcm9ncmVzc19ldmVyeSI6IDUwMDAwLA0KICAgICJlbmNvZGVyX2xlYXJuaW5nX3JhdGUiOiAzZS00LA0KICAgICJoZWFkX2xlYXJuaW5nX3JhdGUiOiAxZS0zLA0KICAgICJ3ZWlnaHRfZGVjYXkiOiAxZS01LA0KICAgICJmZWF0dXJlX3ZvY2FiX3NpemUiOiA4MTkyNCwNCiAgICAiZmVhdHVyZV9lbWJlZF9kaW0iOiA2NCwNCiAgICAiZGVuc2Vfc3RhdGVfZGltIjogMjksDQogICAgImNvbnRleHRfZGltIjogMSwNCiAgICAiaGlzdG9yeV9ldmVudF9kaW0iOiAxNiwNCiAgICAiaGlzdG9yeV9kZWx0YV9kaW0iOiAxMSwNCiAgICAiaGlzdG9yeV9oaWRkZW5fZGltIjogOTYsDQogICAgInNoYXJlZF9oaWRkZW5fZGltIjogMTI4LA0KICAgICJkcm9wb3V0IjogMC4xMCwNCiAgICAiZW5hYmxlX3RocmVhdF9oZWFkIjogVHJ1ZSwNCiAgICAidGhyZWF0X2xvc3Nfd2VpZ2h0IjogMC4yLA0KICAgICJkZXZpY2UiOiAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiLA0KfQ0KDQoNCmRlZiBzcGxpdF9ieV9nYW1lX3RyYWluX3ZhbGlkX3Rlc3QoDQogICAgc2FtcGxlczogTGlzdFtBbnldLCB2YWxpZF9zaXplOiBmbG9hdCwgdGVzdF9zaXplOiBmbG9hdCwgc2VlZDogaW50DQopIC0+IFR1cGxlW0xpc3RbQW55XSwgTGlzdFtBbnldLCBMaXN0W0FueV0sIERpY3Rbc3RyLCBMaXN0W3N0cl1dXToNCiAgICAiIiJTcGxpdCBzYW1wbGVzIGJ5IGdhbWUgaWQgaW50byB0cmFpbi92YWxpZC90ZXN0IHdpdGggZGlzam9pbnQgZ2FtZXMuIiIiDQogICAgZ2FtZV9pZHMgPSBzb3J0ZWQoe3MuZ2FtZV9pZCBmb3IgcyBpbiBzYW1wbGVzfSkNCiAgICBpZiBsZW4oZ2FtZV9pZHMpIDwgMzoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTmVlZCBhdCBsZWFzdCAzIHVuaXF1ZSBnYW1lcyBmb3IgdHJhaW4vdmFsaWQvdGVzdCBzcGxpdCIpDQoNCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpDQogICAgcm5nLnNodWZmbGUoZ2FtZV9pZHMpDQoNCiAgICBuX3RvdGFsID0gbGVuKGdhbWVfaWRzKQ0KICAgIG5fdGVzdCA9IG1heCgxLCBpbnQobl90b3RhbCAqIHRlc3Rfc2l6ZSkpDQogICAgbl92YWxpZCA9IG1heCgxLCBpbnQobl90b3RhbCAqIHZhbGlkX3NpemUpKQ0KDQogICAgaWYgbl90ZXN0ICsgbl92YWxpZCA+PSBuX3RvdGFsOg0KICAgICAgICBuX3Rlc3QgPSAxDQogICAgICAgIG5fdmFsaWQgPSAxDQoNCiAgICB0ZXN0X2dhbWVzID0gc2V0KGdhbWVfaWRzWzpuX3Rlc3RdKQ0KICAgIHZhbGlkX2dhbWVzID0gc2V0KGdhbWVfaWRzW25fdGVzdDpuX3Rlc3QgKyBuX3ZhbGlkXSkNCiAgICB0cmFpbl9nYW1lcyA9IHNldChnYW1lX2lkc1tuX3Rlc3QgKyBuX3ZhbGlkOl0pDQoNCiAgICB0cmFpbiA9IFtzIGZvciBzIGluIHNhbXBsZXMgaWYgcy5nYW1lX2lkIGluIHRyYWluX2dhbWVzXQ0KICAgIHZhbGlkID0gW3MgZm9yIHMgaW4gc2FtcGxlcyBpZiBzLmdhbWVfaWQgaW4gdmFsaWRfZ2FtZXNdDQogICAgdGVzdCA9IFtzIGZvciBzIGluIHNhbXBsZXMgaWYgcy5nYW1lX2lkIGluIHRlc3RfZ2FtZXNdDQoNCiAgICBpZiBub3QgdHJhaW4gb3Igbm90IHZhbGlkIG9yIG5vdCB0ZXN0Og0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTcGxpdCBwcm9kdWNlZCBlbXB0eSB0cmFpbi92YWxpZC90ZXN0IHNldCIpDQoNCiAgICBzcGxpdF9pbmZvID0gew0KICAgICAgICAidHJhaW5fZ2FtZV9pZHMiOiBzb3J0ZWQodHJhaW5fZ2FtZXMpLA0KICAgICAgICAidmFsaWRfZ2FtZV9pZHMiOiBzb3J0ZWQodmFsaWRfZ2FtZXMpLA0KICAgICAgICAidGVzdF9nYW1lX2lkcyI6IHNvcnRlZCh0ZXN0X2dhbWVzKSwNCiAgICB9DQogICAgcmV0dXJuIHRyYWluLCB2YWxpZCwgdGVzdCwgc3BsaXRfaW5mbw0KDQoNCmRlZiBidWlsZF9vcHRpbWl6ZXIobW9kZWw6IEZhY3Rvcml6ZWRQb2xpY3lNb2RlbCwgY29uZmlnOiBEaWN0W3N0ciwgQW55XSkgLT4gdG9yY2gub3B0aW0uT3B0aW1pemVyOg0KICAgICIiIkJ1aWxkIG9wdGltaXplciB3aXRoIHNlcGFyYXRlIGxycyBmb3IgZW5jb2RlcnMgdnMgcG9saWN5IGhlYWRzLiIiIg0KICAgIGVuY29kZXJfcHJlZml4ZXMgPSAoInN0YXRlX2VuY29kZXIuIiwgImhpc3RvcnlfZW5jb2Rlci4iKQ0KDQogICAgZW5jb2Rlcl9wYXJhbXM6IExpc3RbdG9yY2gubm4uUGFyYW1ldGVyXSA9IFtdDQogICAgaGVhZF9wYXJhbXM6IExpc3RbdG9yY2gubm4uUGFyYW1ldGVyXSA9IFtdDQoNCiAgICBmb3IgbmFtZSwgcGFyYW0gaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOg0KICAgICAgICBpZiBub3QgcGFyYW0ucmVxdWlyZXNfZ3JhZDoNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIGlmIG5hbWUuc3RhcnRzd2l0aChlbmNvZGVyX3ByZWZpeGVzKToNCiAgICAgICAgICAgIGVuY29kZXJfcGFyYW1zLmFwcGVuZChwYXJhbSkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIGhlYWRfcGFyYW1zLmFwcGVuZChwYXJhbSkNCg0KICAgIHJldHVybiB0b3JjaC5vcHRpbS5BZGFtVygNCiAgICAgICAgWw0KICAgICAgICAgICAgeyJwYXJhbXMiOiBlbmNvZGVyX3BhcmFtcywgImxyIjogZmxvYXQoY29uZmlnWyJlbmNvZGVyX2xlYXJuaW5nX3JhdGUiXSl9LA0KICAgICAgICAgICAgeyJwYXJhbXMiOiBoZWFkX3BhcmFtcywgImxyIjogZmxvYXQoY29uZmlnWyJoZWFkX2xlYXJuaW5nX3JhdGUiXSl9LA0KICAgICAgICBdLA0KICAgICAgICB3ZWlnaHRfZGVjYXk9ZmxvYXQoY29uZmlnWyJ3ZWlnaHRfZGVjYXkiXSksDQogICAgKQ0KDQoNCmRlZiBtYWluKCkgLT4gTm9uZToNCiAgICAiIiJSdW4gcGxheWVyLXNwZWNpZmljIGFkYXB0YXRpb24gYW5kIHNhdmUgYmVzdCBjaGVja3BvaW50IGJ5IHZhbGlkIG1vdmUgYWNjdXJhY3kuIiIiDQogICAgc2V0X3NlZWQoaW50KENPTkZJR1sicmFuZG9tX3NlZWQiXSkpDQoNCiAgICBpbl9wYXRoID0gUGF0aChDT05GSUdbImlucHV0X2pzb25sIl0pDQogICAgaWYgbm90IGluX3BhdGguZXhpc3RzKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiSW5wdXQgZmlsZSBub3QgZm91bmQ6IHtpbl9wYXRofSIpDQoNCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2Uoc3RyKENPTkZJR1siZGV2aWNlIl0pKQ0KICAgIGlzX2NwdSA9IGRldmljZS50eXBlID09ICJjcHUiDQoNCiAgICBtYXhfc2FtcGxlc19yYXcgPSBDT05GSUcuZ2V0KCJtYXhfc2FtcGxlc19jcHUiKSBpZiBpc19jcHUgZWxzZSBDT05GSUcuZ2V0KCJtYXhfc2FtcGxlcyIpDQogICAgbWF4X3NhbXBsZXMgPSBpbnQobWF4X3NhbXBsZXNfcmF3KSBpZiBtYXhfc2FtcGxlc19yYXcgaXMgbm90IE5vbmUgZWxzZSBOb25lDQoNCiAgICBzYW1wbGVzID0gbG9hZF9zYW1wbGVzKA0KICAgICAgICBpbl9wYXRoLA0KICAgICAgICBtYXhfc2FtcGxlcz1tYXhfc2FtcGxlcywNCiAgICAgICAgcmVzZXJ2b2lyPWJvb2woQ09ORklHLmdldCgicmVzZXJ2b2lyX3NhbXBsZV9mb3JfY2FwIiwgRmFsc2UpKSwNCiAgICAgICAgc2VlZD1pbnQoQ09ORklHWyJyYW5kb21fc2VlZCJdKSwNCiAgICAgICAgcHJvZ3Jlc3NfZXZlcnk9aW50KENPTkZJRy5nZXQoImxvYWRfcHJvZ3Jlc3NfZXZlcnkiLCAwKSksDQogICAgICAgIHByb2dyZXNzX2xhYmVsPWYie0NPTkZJR1snZGF0YXNldF90YWcnXX06bG9hZCIsDQogICAgKQ0KICAgIGlmIG1heF9zYW1wbGVzIGlzIG5vdCBOb25lOg0KICAgICAgICBwcmludChmIkxvYWRlZCBjYXBwZWQgZmluZS10dW5lIHNhbXBsZSBzZXQ6IHtsZW4oc2FtcGxlcyl9IHJvd3MgKG1heF9zYW1wbGVzPXttYXhfc2FtcGxlc30pIikNCiAgICBlbHNlOg0KICAgICAgICBwcmludChmIkxvYWRlZCBmdWxsIGZpbmUtdHVuZSBzYW1wbGUgc2V0OiB7bGVuKHNhbXBsZXMpfSByb3dzIikNCg0KICAgIHRyYWluX3NhbXBsZXMsIHZhbGlkX3NhbXBsZXMsIHRlc3Rfc2FtcGxlcywgc3BsaXRfaW5mbyA9IHNwbGl0X2J5X2dhbWVfdHJhaW5fdmFsaWRfdGVzdCgNCiAgICAgICAgc2FtcGxlcz1zYW1wbGVzLA0KICAgICAgICB2YWxpZF9zaXplPWZsb2F0KENPTkZJR1sidmFsaWRfc2l6ZSJdKSwNCiAgICAgICAgdGVzdF9zaXplPWZsb2F0KENPTkZJR1sidGVzdF9zaXplIl0pLA0KICAgICAgICBzZWVkPWludChDT05GSUdbInJhbmRvbV9zZWVkIl0pLA0KICAgICkNCg0KICAgIHNwbGl0X291dCA9IFBhdGgoQ09ORklHWyJob25lc3Rfc3BsaXRfb3V0cHV0X3BhdGgiXSkNCiAgICBzcGxpdF9vdXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICB3aXRoIHNwbGl0X291dC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOg0KICAgICAgICBqc29uLmR1bXAoc3BsaXRfaW5mbywgaGFuZGxlLCBpbmRlbnQ9MikNCg0KICAgIHByaW50KA0KICAgICAgICAiU3BsaXQgc3VtbWFyeSB8ICINCiAgICAgICAgZiJ0cmFpbl9nYW1lcz17bGVuKHNwbGl0X2luZm9bJ3RyYWluX2dhbWVfaWRzJ10pfSwgIg0KICAgICAgICBmInZhbGlkX2dhbWVzPXtsZW4oc3BsaXRfaW5mb1sndmFsaWRfZ2FtZV9pZHMnXSl9LCAiDQogICAgICAgIGYidGVzdF9nYW1lcz17bGVuKHNwbGl0X2luZm9bJ3Rlc3RfZ2FtZV9pZHMnXSl9Ig0KICAgICkNCiAgICBwcmludChmIkhvbmVzdCBzcGxpdCBzYXZlZCB0bzoge3NwbGl0X291dH0iKQ0KDQogICAgZWZmZWN0aXZlX2Vwb2NocyA9IGludChDT05GSUdbImVwb2Noc19jcHUiXSBpZiBpc19jcHUgZWxzZSBDT05GSUdbImVwb2NocyJdKQ0KICAgIGJhdGNoX3NpemUgPSBpbnQoQ09ORklHWyJiYXRjaF9zaXplX2NwdSJdIGlmIGlzX2NwdSBlbHNlIENPTkZJR1siYmF0Y2hfc2l6ZSJdKQ0KICAgIGV2YWxfYmF0Y2hfc2l6ZSA9IGludChDT05GSUdbImV2YWxfYmF0Y2hfc2l6ZV9jcHUiXSBpZiBpc19jcHUgZWxzZSBDT05GSUdbImV2YWxfYmF0Y2hfc2l6ZSJdKQ0KICAgIGdyYWRfYWNjdW1fc3RlcHMgPSBpbnQoQ09ORklHWyJncmFkX2FjY3VtX3N0ZXBzX2NwdSJdIGlmIGlzX2NwdSBlbHNlIENPTkZJR1siZ3JhZF9hY2N1bV9zdGVwcyJdKQ0KDQogICAgbW9kZWwgPSBGYWN0b3JpemVkUG9saWN5TW9kZWwoQ09ORklHKS50byhkZXZpY2UpDQoNCiAgICBsb2FkZWRfa2V5cyA9IDANCiAgICBpZiBib29sKENPTkZJRy5nZXQoInVzZV9wcmV0cmFpbmVkX2VuY29kZXJzIiwgRmFsc2UpKToNCiAgICAgICAgcHJlX3BhdGggPSBQYXRoKENPTkZJR1sicHJldHJhaW5lZF9wYXRoIl0pDQogICAgICAgIGlmIHByZV9wYXRoLmV4aXN0cygpOg0KICAgICAgICAgICAgbG9hZGVkX2tleXMgPSBsb2FkX3ByZXRyYWluZWRfZW5jb2RlcnMobW9kZWwsIHByZV9wYXRoLCBkZXZpY2U9ZGV2aWNlKQ0KICAgICAgICAgICAgcHJpbnQoZiJMb2FkZWQgcHJldHJhaW5lZCBlbmNvZGVyIHRlbnNvcnM6IHtsb2FkZWRfa2V5c30iKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgcHJpbnQoZiJQcmV0cmFpbmVkIGNoZWNrcG9pbnQgbm90IGZvdW5kLCB0cmFpbmluZyBmcm9tIHNjcmF0Y2g6IHtwcmVfcGF0aH0iKQ0KDQogICAgb3B0aW1pemVyID0gYnVpbGRfb3B0aW1pemVyKG1vZGVsLCBDT05GSUcpDQoNCiAgICBiZXN0X21vdmVfYWNjID0gLTEuMA0KICAgIGJlc3RfbWV0cmljczogRGljdFtzdHIsIEFueV0gPSB7fQ0KDQogICAgZm9yIGVwb2NoIGluIHJhbmdlKDEsIGVmZmVjdGl2ZV9lcG9jaHMgKyAxKToNCiAgICAgICAgdHJhaW5fbG9zcyA9IHRyYWluX29uZV9lcG9jaCgNCiAgICAgICAgICAgIG1vZGVsPW1vZGVsLA0KICAgICAgICAgICAgc2FtcGxlcz10cmFpbl9zYW1wbGVzLA0KICAgICAgICAgICAgb3B0aW1pemVyPW9wdGltaXplciwNCiAgICAgICAgICAgIGRldmljZT1kZXZpY2UsDQogICAgICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsDQogICAgICAgICAgICBwcmludF9ldmVyeT1pbnQoQ09ORklHWyJwcmludF9ldmVyeSJdKSwNCiAgICAgICAgICAgIGVwb2NoX2luZGV4PWVwb2NoLA0KICAgICAgICAgICAgdG90YWxfZXBvY2hzPWVmZmVjdGl2ZV9lcG9jaHMsDQogICAgICAgICAgICBncmFkX2FjY3VtX3N0ZXBzPWdyYWRfYWNjdW1fc3RlcHMsDQogICAgICAgICkNCg0KICAgICAgICB0cmFpbl9tZXRyaWNzID0gZXZhbHVhdGUobW9kZWwsIHRyYWluX3NhbXBsZXMsIGRldmljZT1kZXZpY2UsIGJhdGNoX3NpemU9ZXZhbF9iYXRjaF9zaXplKQ0KICAgICAgICB2YWxpZF9tZXRyaWNzID0gZXZhbHVhdGUobW9kZWwsIHZhbGlkX3NhbXBsZXMsIGRldmljZT1kZXZpY2UsIGJhdGNoX3NpemU9ZXZhbF9iYXRjaF9zaXplKQ0KDQogICAgICAgIHByaW50KA0KICAgICAgICAgICAgZiJFcG9jaCB7ZXBvY2g6MDJkfSBkb25lIHwgIg0KICAgICAgICAgICAgZiJsb3NzPXt0cmFpbl9sb3NzOi40Zn0gfCAiDQogICAgICAgICAgICBmInRyYWluX21vdmVfYWNjPXt0cmFpbl9tZXRyaWNzWydtb3ZlX2FjYyddOi40Zn0gfCAiDQogICAgICAgICAgICBmInZhbGlkX21vdmVfYWNjPXt2YWxpZF9tZXRyaWNzWydtb3ZlX2FjYyddOi40Zn0iDQogICAgICAgICkNCg0KICAgICAgICBpZiB2YWxpZF9tZXRyaWNzWyJtb3ZlX2FjYyJdID4gYmVzdF9tb3ZlX2FjYzoNCiAgICAgICAgICAgIGJlc3RfbW92ZV9hY2MgPSB2YWxpZF9tZXRyaWNzWyJtb3ZlX2FjYyJdDQogICAgICAgICAgICBiZXN0X21ldHJpY3MgPSB7DQogICAgICAgICAgICAgICAgImJlc3RfZXBvY2giOiBlcG9jaCwNCiAgICAgICAgICAgICAgICAidHJhaW4iOiB0cmFpbl9tZXRyaWNzLA0KICAgICAgICAgICAgICAgICJ2YWxpZCI6IHZhbGlkX21ldHJpY3MsDQogICAgICAgICAgICAgICAgImhpc3RvcnlfcGxpZXMiOiBpbnQoQ09ORklHWyJoaXN0b3J5X3BsaWVzIl0pLA0KICAgICAgICAgICAgICAgICJtb2RlIjogImZpbmV0dW5lIiwNCiAgICAgICAgICAgICAgICAibG9hZGVkX3ByZXRyYWluZWRfZW5jb2Rlcl90ZW5zb3JzIjogbG9hZGVkX2tleXMsDQogICAgICAgICAgICAgICAgImhvbmVzdF9zcGxpdF9wYXRoIjogc3RyKHNwbGl0X291dCksDQogICAgICAgICAgICB9DQogICAgICAgICAgICBzYXZlX2NoZWNrcG9pbnQobW9kZWwsIGRpY3QoQ09ORklHKSwgYmVzdF9tZXRyaWNzLCBQYXRoKENPTkZJR1sibW9kZWxfb3V0cHV0X3BhdGgiXSkpDQoNCiAgICAjIEhvbmVzdCB0ZXN0IGV2YWx1YXRpb24gb25seSBvbmNlIG9uIHRoZSBiZXN0IGNoZWNrcG9pbnQuDQogICAgY2twdCA9IGxvYWRfY2hlY2twb2ludChQYXRoKENPTkZJR1sibW9kZWxfb3V0cHV0X3BhdGgiXSksIGRldmljZT1kZXZpY2UpDQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGNrcHRbIm1vZGVsX3N0YXRlX2RpY3QiXSkNCiAgICBtb2RlbC5ldmFsKCkNCiAgICB0ZXN0X21ldHJpY3MgPSBldmFsdWF0ZShtb2RlbCwgdGVzdF9zYW1wbGVzLCBkZXZpY2U9ZGV2aWNlLCBiYXRjaF9zaXplPWV2YWxfYmF0Y2hfc2l6ZSkNCg0KICAgIGJlc3RfbWV0cmljc1sidGVzdCJdID0gdGVzdF9tZXRyaWNzDQogICAgYmVzdF9tZXRyaWNzWyJob25lc3RfdGVzdF9ub3RlIl0gPSAiVGVzdCBnYW1lcyBhcmUgZGlzam9pbnQgZnJvbSB0cmFpbiBhbmQgdmFsaWQgYnkgZ2FtZV9pZC4iDQoNCiAgICBtZXRyaWNzX3BhdGggPSBQYXRoKENPTkZJR1sibWV0cmljc19vdXRwdXRfcGF0aCJdKQ0KICAgIG1ldHJpY3NfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIHdpdGggbWV0cmljc19wYXRoLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBoYW5kbGU6DQogICAgICAgIGpzb24uZHVtcChiZXN0X21ldHJpY3MsIGhhbmRsZSwgaW5kZW50PTIpDQoNCiAgICBwcmludChmIkJlc3QgdmFsaWQgbW92ZV9hY2M6IHtiZXN0X21vdmVfYWNjOi40Zn0iKQ0KICAgIHByaW50KGYiSG9uZXN0IHRlc3QgbW92ZV9hY2M6IHt0ZXN0X21ldHJpY3NbJ21vdmVfYWNjJ106LjRmfSIpDQogICAgcHJpbnQoZiJNb2RlbCBzYXZlZDoge0NPTkZJR1snbW9kZWxfb3V0cHV0X3BhdGgnXX0iKQ0KICAgIHByaW50KGYiTWV0cmljcyBzYXZlZDoge0NPTkZJR1snbWV0cmljc19vdXRwdXRfcGF0aCddfSIpDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICBtYWluKCkNCg0KDQoNCg0K"
}

WORK_ROOT = Path("/kaggle/working/Imitator")
SCRIPTS_DIR = WORK_ROOT / "scripts"
RAW_ROOT = WORK_ROOT / "data" / "raw"
PRETRAIN_ROOT = RAW_ROOT / "pretrain_multi"
FINETUNE_ROOT = RAW_ROOT / "finetune_players"

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_ROOT.mkdir(parents=True, exist_ok=True)
FINETUNE_ROOT.mkdir(parents=True, exist_ok=True)

for rel_path, encoded in EMBEDDED_FILES.items():
    out_path = WORK_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(base64.b64decode(encoded.encode("ascii")))

SHARDED_PRETRAIN_SCRIPT = r'''
#!/usr/bin/env python3
from __future__ import annotations

import hashlib
import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import torch

from history_policy_lib import PolicySample, FactorizedPolicyModel, batch_loss, collate_batch, evaluate, format_eta, save_checkpoint, set_seed
from pipeline_config import HISTORY_PLIES, PRETRAIN_DATASET_TAG, TARGET_USERNAME, models_dir, processed_dir


def normalize_player_id(name: str) -> str:
    return " ".join(name.strip().split()).lower()


CONFIG: Dict[str, Any] = {
    "dataset_tag": PRETRAIN_DATASET_TAG,
    "history_plies": HISTORY_PLIES,
    "input_jsonl_dir": processed_dir(PRETRAIN_DATASET_TAG) / "_parallel_parts",
    "model_output_path": models_dir(PRETRAIN_DATASET_TAG) / "history_policy_pretrain_stream.pt",
    "metrics_output_path": models_dir(PRETRAIN_DATASET_TAG) / "history_policy_pretrain_stream_metrics.json",
    "valid_size": 0.10,
    "random_seed": 42,
    "epochs": 8,
    "epochs_cpu": 4,
    "batch_size": 256,
    "batch_size_cpu": 48,
    "print_every": 20,
    "eval_batch_size": 256,
    "eval_batch_size_cpu": 128,
    "grad_accum_steps": 1,
    "grad_accum_steps_cpu": 2,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "feature_vocab_size": 81924,
    "feature_embed_dim": 64,
    "dense_state_dim": 29,
    "context_dim": 1,
    "history_event_dim": 16,
    "history_delta_dim": 11,
    "history_hidden_dim": 96,
    "shared_hidden_dim": 128,
    "dropout": 0.10,
    "enable_threat_head": True,
    "threat_loss_weight": 0.2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "strict_target_isolation": False,
    "excluded_player_ids": [normalize_player_id(TARGET_USERNAME)],
    "shuffle_buffer_size": 20000,
    "scan_progress_every": 50000,
    "stream_progress_every": 100000,
    "train_eval_max_samples": 50000,
    "valid_eval_max_samples": 50000,
}


def iter_input_files(input_dir: Path) -> List[Path]:
    return sorted(p for p in input_dir.glob("*.jsonl") if p.is_file())


def iter_jsonl_paths(paths: List[Path]) -> Iterable[Dict[str, Any]]:
    for path in paths:
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if line:
                    yield json.loads(line)


def row_to_sample(row: Dict[str, Any]) -> PolicySample:
    return PolicySample(
        game_id=str(row.get("game_id", "")),
        player_id=str(row.get("player_id", "unknown")),
        active_feature_indices=[int(x) for x in row["active_feature_indices"]],
        dense_state=[float(x) for x in row["dense_state"]],
        history_event=[[float(v) for v in x] for x in row["history_event"]],
        history_delta=[[float(v) for v in x] for x in row["history_delta"]],
        history_mask=[int(x) for x in row.get("history_mask", [1] * len(row["history_event"]))],
        context=[float(x) for x in row["context"]],
        piece_slot_to_square=[int(x) for x in row.get("piece_slot_to_square", [-1] * 16)],
        legal_piece_slot_mask=[int(x) for x in row.get("legal_piece_slot_mask", [0] * 16)],
        legal_from_mask=[int(x) for x in row["legal_from_mask"]],
        legal_to_by_from={str(k): [int(v) for v in vals] for k, vals in row["legal_to_by_from"].items()},
        target_piece_slot=int(row.get("target_piece_slot", 0)),
        target_from_sq=int(row["target_from_sq"]),
        target_to_sq=int(row["target_to_sq"]),
        target_promotion=int(row.get("target_promotion", 0)),
        target_under_threat=int(row.get("target_under_threat", 0)),
    )


def keep_sample(player_id: str, config: Dict[str, Any]) -> bool:
    if not bool(config.get("strict_target_isolation", False)):
        return True
    excluded = {str(x).strip().lower() for x in config.get("excluded_player_ids", []) if str(x).strip()}
    return str(player_id).strip().lower() not in excluded


def split_hash_is_valid(game_id: str, valid_size: float, seed: int) -> bool:
    key = f"{seed}:{game_id}".encode("utf-8")
    bucket = int.from_bytes(hashlib.sha1(key).digest()[:8], byteorder="big")
    return (bucket / float(2**64)) < valid_size


def reservoir_push(reservoir: List[PolicySample], sample: PolicySample, cap: int, seen: int, rng: random.Random) -> None:
    if cap <= 0:
        return
    if len(reservoir) < cap:
        reservoir.append(sample)
        return
    j = rng.randrange(seen)
    if j < cap:
        reservoir[j] = sample


def scan_stream_metadata(config: Dict[str, Any]) -> Tuple[Dict[str, int], List[PolicySample], List[PolicySample]]:
    files = iter_input_files(Path(config["input_jsonl_dir"]))
    valid_size = float(config["valid_size"])
    seed = int(config["random_seed"])
    progress_every = int(config.get("scan_progress_every", 0))
    train_eval_cap = int(config.get("train_eval_max_samples", 0) or 0)
    valid_eval_cap = int(config.get("valid_eval_max_samples", 0) or 0)
    train_eval: List[PolicySample] = []
    valid_eval: List[PolicySample] = []
    train_eval_seen = 0
    valid_eval_seen = 0
    counts = {"rows_total": 0, "rows_kept": 0, "rows_filtered_out": 0, "train_rows": 0, "valid_rows": 0}
    rng = random.Random(seed + 17)
    start = time.time()
    for row in iter_jsonl_paths(files):
        counts["rows_total"] += 1
        player_id = str(row.get("player_id", "unknown"))
        if not keep_sample(player_id, config):
            counts["rows_filtered_out"] += 1
            continue
        counts["rows_kept"] += 1
        game_id = str(row.get("game_id", ""))
        is_valid = split_hash_is_valid(game_id, valid_size, seed)
        sample = row_to_sample(row)
        if is_valid:
            counts["valid_rows"] += 1
            valid_eval_seen += 1
            reservoir_push(valid_eval, sample, valid_eval_cap, valid_eval_seen, rng)
        else:
            counts["train_rows"] += 1
            train_eval_seen += 1
            reservoir_push(train_eval, sample, train_eval_cap, train_eval_seen, rng)
        if progress_every > 0 and counts["rows_total"] % progress_every == 0:
            elapsed = time.time() - start
            rate = counts["rows_total"] / max(elapsed, 1e-6)
            print(f"[scan] rows={counts['rows_total']:,} kept={counts['rows_kept']:,} train={counts['train_rows']:,} valid={counts['valid_rows']:,} speed={rate:,.0f} rows/s")
    if counts["train_rows"] == 0 or counts["valid_rows"] == 0:
        raise ValueError("Deterministic split produced empty train or valid set")
    return counts, train_eval, valid_eval


def iter_train_batches_stream(config: Dict[str, Any], epoch_seed: int, batch_size: int) -> Iterable[List[PolicySample]]:
    files = iter_input_files(Path(config["input_jsonl_dir"]))
    valid_size = float(config["valid_size"])
    seed = int(config["random_seed"])
    buffer_size = max(batch_size, int(config.get("shuffle_buffer_size", 20000)))
    rng = random.Random(epoch_seed)
    buffer: List[PolicySample] = []
    carry: List[PolicySample] = []
    for row in iter_jsonl_paths(files):
        player_id = str(row.get("player_id", "unknown"))
        if not keep_sample(player_id, config):
            continue
        game_id = str(row.get("game_id", ""))
        if split_hash_is_valid(game_id, valid_size, seed):
            continue
        buffer.append(row_to_sample(row))
        if len(buffer) >= buffer_size:
            rng.shuffle(buffer)
            mixed = carry + buffer
            full = (len(mixed) // batch_size) * batch_size
            for i in range(0, full, batch_size):
                yield mixed[i:i + batch_size]
            carry = mixed[full:]
            buffer = []
    mixed = carry + buffer
    for i in range(0, len(mixed), batch_size):
        yield mixed[i:i + batch_size]


def train_one_epoch_stream(model, optimizer, device, config, epoch_index, total_epochs, batch_size, grad_accum_steps, n_train_rows):
    model.train()
    progress_every = int(config.get("print_every", 20))
    n_batches = max(1, math.ceil(n_train_rows / batch_size))
    total_loss = 0.0
    step = 0
    rows_seen = 0
    start = time.time()
    optimizer.zero_grad(set_to_none=True)
    for batch_samples in iter_train_batches_stream(config, int(config["random_seed"]) + epoch_index, batch_size):
        if not batch_samples:
            continue
        step += 1
        rows_seen += len(batch_samples)
        batch = collate_batch(batch_samples, device=device)
        loss = batch_loss(model, batch)
        loss_value = float(loss.item())
        if not math.isfinite(loss_value):
            optimizer.zero_grad(set_to_none=True)
            continue
        (loss / grad_accum_steps).backward()
        if step % grad_accum_steps == 0 or step == n_batches:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += loss_value
        if step % progress_every == 0 or step == n_batches:
            elapsed = time.time() - start
            rate = step / max(elapsed, 1e-6)
            eta = (n_batches - step) / max(rate, 1e-6)
            print(f"Epoch {epoch_index:02d}/{total_epochs:02d} | step {step:4d}/{n_batches:4d} | rows={rows_seen:,}/{n_train_rows:,} | loss={total_loss/step:.4f} | eta={format_eta(eta)}")
    if step == 0:
        raise ValueError("No training batches were produced")
    return total_loss / step


def main() -> None:
    set_seed(int(CONFIG["random_seed"]))
    input_dir = Path(CONFIG["input_jsonl_dir"])
    if not input_dir.exists():
        raise FileNotFoundError(f"Input shard directory not found: {input_dir}")
    device = torch.device(str(CONFIG["device"]))
    is_cpu = device.type == "cpu"
    effective_epochs = int(CONFIG["epochs_cpu"] if is_cpu else CONFIG["epochs"])
    batch_size = int(CONFIG["batch_size_cpu"] if is_cpu else CONFIG["batch_size"])
    eval_batch_size = int(CONFIG["eval_batch_size_cpu"] if is_cpu else CONFIG["eval_batch_size"])
    grad_accum_steps = int(CONFIG["grad_accum_steps_cpu"] if is_cpu else CONFIG["grad_accum_steps"])
    counts, train_eval_samples, valid_eval_samples = scan_stream_metadata(CONFIG)
    model = FactorizedPolicyModel(CONFIG).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(CONFIG["learning_rate"]), weight_decay=float(CONFIG["weight_decay"]))
    best_move_acc = -1.0
    best_metrics: Dict[str, Any] = {}
    for epoch in range(1, effective_epochs + 1):
        train_loss = train_one_epoch_stream(model, optimizer, device, CONFIG, epoch, effective_epochs, batch_size, grad_accum_steps, int(counts["train_rows"]))
        train_metrics = evaluate(model, train_eval_samples, device=device, batch_size=eval_batch_size)
        valid_metrics = evaluate(model, valid_eval_samples, device=device, batch_size=eval_batch_size)
        print(f"Epoch {epoch:02d} done | loss={train_loss:.4f} | train_move_acc={train_metrics['move_acc']:.4f} | valid_move_acc={valid_metrics['move_acc']:.4f}")
        if valid_metrics["move_acc"] > best_move_acc:
            best_move_acc = valid_metrics["move_acc"]
            best_metrics = {"best_epoch": epoch, "train": train_metrics, "valid": valid_metrics, "history_plies": int(CONFIG["history_plies"]), "mode": "pretrain_stream_sharded", "strict_target_isolation": bool(CONFIG.get("strict_target_isolation", False)), "stream_counts": counts, "train_eval_reservoir_size": len(train_eval_samples), "valid_eval_reservoir_size": len(valid_eval_samples)}
            save_checkpoint(model, dict(CONFIG), best_metrics, Path(CONFIG["model_output_path"]))
    metrics_path = Path(CONFIG["metrics_output_path"])
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    with metrics_path.open("w", encoding="utf-8") as handle:
        json.dump(best_metrics, handle, indent=2)


if __name__ == "__main__":
    main()

'''

(WORK_ROOT / "scripts" / "script3_pretrain_history_policy_sharded.py").write_text(SHARDED_PRETRAIN_SCRIPT, encoding="utf-8")

def normalize_slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(text).strip().lower()).strip("_")


def copy_tree(src: Path, dst: Path) -> int:
    copied = 0
    for path in src.rglob("*"):
        if not path.is_file():
            continue
        rel = path.relative_to(src)
        out = dst / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, out)
        copied += 1
    return copied


def discover_data_root() -> Path:
    explicit = Path(KAGGLE_INPUT_DATA_DIR)
    if explicit.exists():
        return explicit
    fallback = Path("/kaggle/input")
    for data_dir in fallback.rglob("data"):
        if (data_dir / "twic").exists() and (data_dir / "target").exists():
            return data_dir.parent
    raise FileNotFoundError("Could not find raw chess data root under /kaggle/input")


data_dataset_root = discover_data_root()
data_root = data_dataset_root / "data"
print("data dataset root:", data_dataset_root)
print("data root:", data_root)

stats = {"twic_files": 0, "target_files": 0}
if (data_root / "twic").exists():
    stats["twic_files"] = copy_tree(data_root / "twic", PRETRAIN_ROOT)

target_slug = normalize_slug(DATASET_TAG)
if (data_root / "target").exists():
    for child in (data_root / "target").iterdir():
        if child.is_dir() and normalize_slug(child.name) == target_slug:
            stats["target_files"] = copy_tree(child, FINETUNE_ROOT / DATASET_TAG)
            break

print("materialized files:", len(EMBEDDED_FILES))
print("hydration stats:", json.dumps(stats, indent=2))
print("pretrain pgns:", len(list(PRETRAIN_ROOT.rglob('*.pgn'))))
print("target pgns:", len(list((FINETUNE_ROOT / DATASET_TAG).rglob('*.pgn'))))


In [ ]:
import json
import re
from pathlib import Path

profile = PROFILE_CONFIGS[MODEL_PROFILE]
HISTORY_PLIES = int(profile["history_plies"])


def replace_assignment(text: str, name: str, value_literal: str) -> str:
    pattern = rf"^{re.escape(name)}\s*=\s*.*$"
    replacement = f"{name} = {value_literal}"
    new_text, count = re.subn(pattern, replacement, text, count=1, flags=re.MULTILINE)
    if count != 1:
        raise RuntimeError(f"Could not patch assignment for {name}")
    return new_text


def replace_config_value(text: str, key: str, value_literal: str) -> str:
    pattern = rf'^(\s*"{re.escape(key)}":\s*).*$'
    replacement = rf'\g<1>{value_literal},'
    new_text, count = re.subn(pattern, replacement, text, count=1, flags=re.MULTILINE)
    if count != 1:
        raise RuntimeError(f"Could not patch CONFIG key {key}")
    return new_text


def patch_pipeline_config() -> None:
    path = WORK_ROOT / "scripts" / "pipeline_config.py"
    text = path.read_text(encoding="utf-8")
    text = replace_assignment(text, "DATASET_TAG", json.dumps(DATASET_TAG))
    text = replace_assignment(text, "TARGET_USERNAME", json.dumps(TARGET_USERNAME))
    text = replace_assignment(text, "TARGET_NAME_ALIASES: list[str]", repr(TARGET_NAME_ALIASES))
    text = replace_assignment(text, "ENABLE_STAGE0_MERGE", "False")
    text = replace_assignment(text, "HISTORY_PLIES", str(HISTORY_PLIES))
    text = replace_assignment(text, "USE_STREAM_PRETRAIN_STAGE3", "True" if USE_STREAM_PRETRAIN_STAGE3 else "False")
    path.write_text(text, encoding="utf-8")


COMMON_MODEL_UPDATES = {
    "feature_embed_dim": str(int(profile["feature_embed_dim"])),
    "history_hidden_dim": str(int(profile["history_hidden_dim"])),
    "shared_hidden_dim": str(int(profile["shared_hidden_dim"])),
    "dropout": repr(float(profile["dropout"])),
}


def patch_pretrain_stream_config() -> None:
    path = WORK_ROOT / "scripts" / "script3_pretrain_history_policy_stream.py"
    text = path.read_text(encoding="utf-8")
    updates = {
        "history_plies": str(HISTORY_PLIES),
        "epochs": str(int(profile["pretrain_epochs"])),
        "batch_size": str(int(profile["pretrain_batch_size"])),
        "print_every": str(PRETRAIN_PRINT_EVERY),
        "eval_batch_size": str(int(profile["pretrain_eval_batch_size"])),
        "grad_accum_steps": str(int(profile["pretrain_grad_accum_steps"])),
        "learning_rate": repr(PRETRAIN_LEARNING_RATE),
        "weight_decay": repr(PRETRAIN_WEIGHT_DECAY),
        "strict_target_isolation": "True" if STRICT_TARGET_ISOLATION else "False",
        "shuffle_buffer_size": str(int(profile["pretrain_shuffle_buffer_size"])),
        "scan_progress_every": str(PRETRAIN_SCAN_PROGRESS_EVERY),
        "stream_progress_every": str(PRETRAIN_STREAM_PROGRESS_EVERY),
        "train_eval_max_samples": str(int(profile["pretrain_train_eval_max_samples"])),
        "valid_eval_max_samples": str(int(profile["pretrain_valid_eval_max_samples"])),
    }
    updates.update(COMMON_MODEL_UPDATES)
    for key, value in updates.items():
        text = replace_config_value(text, key, value)
    path.write_text(text, encoding="utf-8")


def patch_pretrain_standard_config() -> None:
    path = WORK_ROOT / "scripts" / "script3_pretrain_history_policy.py"
    text = path.read_text(encoding="utf-8")
    updates = {
        "history_plies": str(HISTORY_PLIES),
        "epochs": str(int(profile["pretrain_epochs"])),
        "batch_size": str(int(profile["pretrain_batch_size"])),
        "print_every": str(PRETRAIN_PRINT_EVERY),
        "eval_batch_size": str(int(profile["pretrain_eval_batch_size"])),
        "grad_accum_steps": str(int(profile["pretrain_grad_accum_steps"])),
        "learning_rate": repr(PRETRAIN_LEARNING_RATE),
        "weight_decay": repr(PRETRAIN_WEIGHT_DECAY),
        "strict_target_isolation": "True" if STRICT_TARGET_ISOLATION else "False",
        "train_eval_max_samples": str(int(profile["pretrain_train_eval_max_samples"])),
        "valid_eval_max_samples": str(int(profile["pretrain_valid_eval_max_samples"])),
    }
    updates.update(COMMON_MODEL_UPDATES)
    for key, value in updates.items():
        text = replace_config_value(text, key, value)
    path.write_text(text, encoding="utf-8")


def patch_finetune_config() -> None:
    path = WORK_ROOT / "scripts" / "script4_finetune_history_policy.py"
    text = path.read_text(encoding="utf-8")
    pretrained_filename = "history_policy_pretrain_stream.pt" if USE_STREAM_PRETRAIN_STAGE3 else "history_policy_pretrain.pt"
    updates = {
        "dataset_tag": json.dumps(DATASET_TAG),
        "history_plies": str(HISTORY_PLIES),
        "pretrained_path": f'models_dir("pretrain_multi") / "{pretrained_filename}"',
        "use_pretrained_encoders": "True" if FINETUNE_USE_PRETRAINED_ENCODERS else "False",
        "epochs": str(int(profile["finetune_epochs"])),
        "batch_size": str(int(profile["finetune_batch_size"])),
        "print_every": str(FINETUNE_PRINT_EVERY),
        "eval_batch_size": str(int(profile["finetune_eval_batch_size"])),
        "grad_accum_steps": str(int(profile["finetune_grad_accum_steps"])),
        "encoder_learning_rate": repr(float(profile["encoder_learning_rate"])),
        "head_learning_rate": repr(float(profile["head_learning_rate"])),
        "weight_decay": repr(float(profile["weight_decay"])),
    }
    updates.update(COMMON_MODEL_UPDATES)
    for key, value in updates.items():
        text = replace_config_value(text, key, value)
    path.write_text(text, encoding="utf-8")


def patch_pretrain_sharded_config() -> None:
    path = WORK_ROOT / "scripts" / "script3_pretrain_history_policy_sharded.py"
    text = path.read_text(encoding="utf-8")
    updates = {
        "history_plies": str(HISTORY_PLIES),
        "epochs": str(int(profile["pretrain_epochs"])),
        "batch_size": str(int(profile["pretrain_batch_size"])),
        "print_every": str(PRETRAIN_PRINT_EVERY),
        "eval_batch_size": str(int(profile["pretrain_eval_batch_size"])),
        "grad_accum_steps": str(int(profile["pretrain_grad_accum_steps"])),
        "learning_rate": repr(PRETRAIN_LEARNING_RATE),
        "weight_decay": repr(PRETRAIN_WEIGHT_DECAY),
        "strict_target_isolation": "True" if STRICT_TARGET_ISOLATION else "False",
        "shuffle_buffer_size": str(int(profile["pretrain_shuffle_buffer_size"])),
        "scan_progress_every": str(PRETRAIN_SCAN_PROGRESS_EVERY),
        "stream_progress_every": str(PRETRAIN_STREAM_PROGRESS_EVERY),
        "train_eval_max_samples": str(int(profile["pretrain_train_eval_max_samples"])),
        "valid_eval_max_samples": str(int(profile["pretrain_valid_eval_max_samples"])),
    }
    updates.update(COMMON_MODEL_UPDATES)
    for key, value in updates.items():
        text = replace_config_value(text, key, value)
    path.write_text(text, encoding="utf-8")


patch_pipeline_config()
patch_pretrain_stream_config()
patch_pretrain_standard_config()
patch_pretrain_sharded_config()
patch_finetune_config()
print("Applied profile:", MODEL_PROFILE)
print(json.dumps(profile, indent=2))


In [ ]:
import importlib
import json
import math
import os
import shutil
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

os.chdir(SCRIPTS_DIR)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import pipeline_config


def _clear_modules() -> None:
    for name in list(sys.modules):
        if name in {
            "pipeline_config",
            "script1_parse_multi_player_positions",
            "script1_parse_pgn_to_positions",
            "script2_encode_policy_samples",
        }:
            del sys.modules[name]


def preprocess_one_pretrain(src_path: str, out_path: str, history_plies: int) -> tuple[str, int]:
    from pathlib import Path
    import json
    from script1_parse_multi_player_positions import iter_all_player_positions
    from script2_encode_policy_samples import encode_row

    src = Path(src_path)
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with out.open("w", encoding="utf-8") as handle:
        for row in iter_all_player_positions(src, history_plies):
            sample = encode_row(row, history_plies)
            if sample is None:
                continue
            handle.write(json.dumps(sample, ensure_ascii=False) + "\n")
            count += 1
    return src.name, count


def preprocess_one_target(src_path: str, out_path: str, target_username: str, aliases: list[str], history_plies: int) -> tuple[str, int]:
    from pathlib import Path
    import json
    from script1_parse_pgn_to_positions import iter_target_player_positions
    from script2_encode_policy_samples import encode_row

    src = Path(src_path)
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with out.open("w", encoding="utf-8") as handle:
        for row in iter_target_player_positions(src, target_username, aliases, history_plies):
            sample = encode_row(row, history_plies)
            if sample is None:
                continue
            handle.write(json.dumps(sample, ensure_ascii=False) + "\n")
            count += 1
    return src.name, count


def disk_free_gb(path: Path) -> float:
    usage = shutil.disk_usage(path)
    return usage.free / (1024 ** 3)


def concat_jsonl(parts: list[Path], out_path: Path) -> int:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    total = 0
    with out_path.open("w", encoding="utf-8") as dst:
        for part in sorted(parts):
            if not part.exists():
                continue
            with part.open("r", encoding="utf-8") as src:
                for line in src:
                    if line.strip():
                        dst.write(line)
                        total += 1
    return total


def run_parallel_jobs(kind: str, files: list[Path], worker_fn, worker_args_builder, tmp_dir: Path, workers: int) -> tuple[list[Path], int]:
    tmp_dir.mkdir(parents=True, exist_ok=True)
    if not files:
        return [], 0
    parts = []
    total_rows = 0
    max_workers = max(1, min(workers, len(files)))
    print(f"{kind}: files={len(files)} workers={max_workers}")
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = []
        for idx, path in enumerate(files):
            out_path = tmp_dir / f"part_{idx:04d}_{path.stem}.jsonl"
            parts.append(out_path)
            futures.append(ex.submit(worker_fn, *worker_args_builder(path, out_path)))
        done = 0
        for fut in as_completed(futures):
            name, count = fut.result()
            total_rows += count
            done += 1
            print(f"[{kind}] done {done}/{len(files)} file={name} rows={count}")
    return parts, total_rows


history_plies = HISTORY_PLIES
workers = max(1, PREPROCESS_WORKERS)
processed_pretrain = pipeline_config.processed_dir("pretrain_multi") / "policy_samples.jsonl"
processed_target = pipeline_config.processed_dir(DATASET_TAG) / "policy_samples.jsonl"

pretrain_files = sorted(p for p in PRETRAIN_ROOT.rglob("*.pgn") if p.is_file())
target_root = FINETUNE_ROOT / DATASET_TAG
target_files = sorted(p for p in target_root.rglob("*.pgn") if p.is_file())

if RUN_PRETRAIN and not pretrain_files:
    RUN_PRETRAIN = False
    FINETUNE_USE_PRETRAINED_ENCODERS = False
    print("Auto-disabled pretraining because no pretrain PGNs were found after hydration.")

if RUN_FINETUNE and not target_files:
    raise FileNotFoundError(f"No target PGNs found under {target_root}")

print(f"disk free before preprocessing: {disk_free_gb(WORK_ROOT):.2f} GB")

if RUN_PRETRAIN:
    tmp_pre = pipeline_config.processed_dir("pretrain_multi") / "_parallel_parts"
    if tmp_pre.exists():
        shutil.rmtree(tmp_pre, ignore_errors=True)
    parts, rows = run_parallel_jobs(
        "pretrain",
        pretrain_files,
        preprocess_one_pretrain,
        lambda path, out_path: (str(path), str(out_path), history_plies),
        tmp_pre,
        workers,
    )
    print(f"pretrain shard rows written: {rows} in {tmp_pre}")
    print(f"pretrain shard count: {len(parts)}")
    print(f"disk free after pretrain preprocessing: {disk_free_gb(WORK_ROOT):.2f} GB")

if RUN_FINETUNE:
    tmp_ft = pipeline_config.processed_dir(DATASET_TAG) / "_parallel_parts"
    parts, rows = run_parallel_jobs(
        "finetune",
        target_files,
        preprocess_one_target,
        lambda path, out_path: (str(path), str(out_path), TARGET_USERNAME, TARGET_NAME_ALIASES, history_plies),
        tmp_ft,
        workers,
    )
    total = concat_jsonl(parts, processed_target)
    shutil.rmtree(tmp_ft, ignore_errors=True)
    print(f"finetune rows written: {total} -> {processed_target}")
    print(f"disk free after finetune preprocessing: {disk_free_gb(WORK_ROOT):.2f} GB")


In [ ]:
import importlib
import os
import sys

os.chdir(SCRIPTS_DIR)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

for name in list(sys.modules):
    if name.startswith(("pipeline_config", "script3_", "script4_", "history_policy_lib")):
        del sys.modules[name]

if RUN_PRETRAIN:
    if USE_STREAM_PRETRAIN_STAGE3:
        pretrain_mod = importlib.import_module("script3_pretrain_history_policy_sharded")
    else:
        pretrain_mod = importlib.import_module("script3_pretrain_history_policy")
    pretrain_mod.main()

if RUN_FINETUNE:
    finetune_mod = importlib.import_module("script4_finetune_history_policy")
    finetune_mod.main()


In [ ]:
import json
from pathlib import Path

artifact_paths = [
    WORK_ROOT / "models" / "pretrain_multi" / "history_policy_pretrain_stream.pt",
    WORK_ROOT / "models" / "pretrain_multi" / "history_policy_pretrain_stream_metrics.json",
    WORK_ROOT / "models" / DATASET_TAG / "history_policy.pt",
    WORK_ROOT / "models" / DATASET_TAG / "history_policy_metrics.json",
    WORK_ROOT / "models" / DATASET_TAG / "honest_split_game_ids.json",
]

for path in artifact_paths:
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path} | {size_mb:.1f} MB")
        if path.suffix == ".json":
            payload = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(payload, indent=2)[:4000])
    else:
        print(f"Missing: {path}")


In [ ]:
import shutil
from pathlib import Path

if EXPORT_ARTIFACTS:
    export_root = Path("/kaggle/working") / f"{EXPORT_NAME}_staging"
    if export_root.exists():
        shutil.rmtree(export_root)
    export_root.mkdir(parents=True, exist_ok=True)

    for rel in [Path("models"), Path("outputs")]:
        src = WORK_ROOT / rel
        if src.exists():
            shutil.copytree(src, export_root / rel)

    if EXPORT_PROCESSED:
        processed_src = WORK_ROOT / "data" / "processed"
        if processed_src.exists():
            shutil.copytree(processed_src, export_root / "data" / "processed")

    archive_base = Path("/kaggle/working") / EXPORT_NAME
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=export_root)
    print(f"Created archive: {archive_path}")
else:
    print("EXPORT_ARTIFACTS is disabled.")
